<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 45%,#00A86A 100%);border-radius:16px;padding:36px 40px;color:#ffffff;font-family:Calibri,'Segoe UI',sans-serif;">
  <div style="font-size:12px;font-weight:700;letter-spacing:3px;text-transform:uppercase;color:#F5C242;">01 &middot; HANDS-ON &middot; DONN&Eacute;ES NUM&Eacute;RIQUES</div>
  <div style="font-size:42px;font-weight:700;line-height:1.08;margin-top:12px;">Ookla &agrave; l'&eacute;chelle avec Elasticsearch</div>
  <div style="font-size:17px;font-style:italic;color:#E6F6EE;margin-top:10px;max-width:820px;">
    Indexer les tuiles de performance Ookla, comprendre le <em>mapping</em> et le type <code style="color:#F5C242;background:rgba(255,255,255,.08);padding:1px 6px;border-radius:4px;">geo_shape</code>, croiser la connectivit&eacute; avec la population WorldPop, puis publier un tableau de bord Kibana pr&ecirc;t &agrave; pr&eacute;senter.
  </div>
  <div style="height:6px;width:140px;background:#F5C242;border-radius:3px;margin-top:26px;"></div>
</div>

<div style="display:flex;gap:14px;margin-top:18px;font-family:Calibri,'Segoe UI',sans-serif;flex-wrap:wrap;">
  <div style="flex:1;min-width:210px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:12px;padding:16px 18px;">
    <div style="font-size:10.5px;font-weight:700;letter-spacing:2px;color:#00704A;text-transform:uppercase;">Environnements</div>
    <div style="font-size:14px;color:#231F20;margin-top:6px;">Google Colab &middot; Kaggle &middot; Local (Linux / WSL / Docker)</div>
  </div>
  <div style="flex:1;min-width:210px;background:#F4F7F5;border:1px solid #D5DED9;border-radius:12px;padding:16px 18px;">
    <div style="font-size:10.5px;font-weight:700;letter-spacing:2px;color:#00704A;text-transform:uppercase;">Dur&eacute;e estim&eacute;e</div>
    <div style="font-size:14px;color:#231F20;margin-top:6px;">45 &agrave; 70 minutes (dont ~8 min d'installation)</div>
  </div>
  <div style="flex:1;min-width:210px;background:#E8F5EF;border:1.5px solid #00A86A;border-radius:12px;padding:16px 18px;">
    <div style="font-size:10.5px;font-weight:700;letter-spacing:2px;color:#00704A;text-transform:uppercase;">Livrable</div>
    <div style="font-size:14px;color:#231F20;margin-top:6px;">2 index Elasticsearch + 1 dashboard Kibana de 13 panneaux, filtrable dans le temps</div>
  </div>
</div>

## Sommaire

| # | Section | Ce que vous faites |
|---|---|---|
| **00** | Param&egrave;tres & charte | Choisir le pays, le trimestre, la version de la stack |
| **01** | Environnement | D&eacute;tection Colab / Kaggle / local, installation des d&eacute;pendances |
| **02** | Cluster Elasticsearch + Kibana | T&eacute;l&eacute;chargement, configuration, d&eacute;marrage, contr&ocirc;le de sant&eacute; |
| **03** | Th&eacute;orie du *quadkey* | Quadtree Web Mercator, pr&eacute;fixes, ordre lexicographique |
| **04** | Emprise & t&eacute;l&eacute;chargement | geoBoundaries ADM0/ADM1, extraction DuckDB sur **N trimestres**, d&eacute;coupe |
| **05** | Population WorldPop | Raster 1 km, densit&eacute; par tuile, statistiques zonales |
| **06** | Mapping & indexation | `geo_shape`, `geo_point`, BKD-tree, `_bulk` |
| **07** | Agr&eacute;gations | `stats`, `percentiles`, `weighted_avg`, **`date_histogram`**, `geotile_grid`, `geo_bounding_box`, `geo_shape`, `geo_distance` |
| **08** | Indice de fracture num&eacute;rique | Calcul, sauvegarde dans un index d'analyse |
| **09** | Dashboard Kibana | 13 panneaux g&eacute;n&eacute;r&eacute;s par code, filtre temporel actif, import tol&eacute;rant aux &eacute;checs |
| **10** | Acc&egrave;s, export, nettoyage | URL publique, NDJSON r&eacute;utilisable, arr&ecirc;t de la stack |

> **Note de style** &mdash; ce notebook suit la charte *AfDB-inspired Institutional* (vert `#00A86A`, vert profond `#00704A`, or `#F5C242`). Ce n'est pas la charte officielle du Groupe de la Banque africaine de d&eacute;veloppement&nbsp;: validez couleurs, polices et logo aupr&egrave;s du d&eacute;partement communication avant tout usage officiel.

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">00 &middot; PARAM&Egrave;TRES</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">La seule cellule que vous devez modifier</div>
</div>

Le pays est un **param&egrave;tre**&nbsp;: changez `COUNTRY_ISO3` et relancez le notebook de bout en bout.
Exemples&nbsp;: `TUN`, `MAR`, `DZA`, `EGY`, `SEN`, `CIV`, `GHA`, `NGA`, `KEN`, `RWA`, `ZAF`, `ETH`.

In [ ]:
# =============================================================================
#  PARAM&Egrave;TRES DU NOTEBOOK
# =============================================================================
COUNTRY_ISO3   = "TUN"            # Code ISO 3166-1 alpha-3 du pays etudie
YEAR           = 2024             # Dernier trimestre vise : annee
QUARTER        = 2                # Dernier trimestre vise : trimestre (1..4)
N_QUARTERS     = 4                # Nombre de trimestres consecutifs a charger (axe de temps)
NETWORK_TYPES  = ["fixed", "mobile"]   # fixed et/ou mobile
WORLDPOP_YEAR  = 2020             # Millesime WorldPop (produit 1 km UN-adjusted)

# ---- Stack Elastic -----------------------------------------------------------
INSTALL_STACK  = True             # False si vous avez deja un cluster joignable
ES_VERSION     = "8.15.3"         # Version d'Elasticsearch ET de Kibana
ES_HOST, ES_PORT = "127.0.0.1", 9200
KIBANA_PORT    = 5601
ES_HEAP        = "1g"             # Taille du tas JVM (1g suffit pour ce TP)

# ---- Garde-fous --------------------------------------------------------------
MAX_TILES_PER_PERIOD = 500_000    # Plafond de securite par (reseau, trimestre)
BULK_CHUNK         = 2_000        # Taille des lots d'indexation

# ---- Noms des index ----------------------------------------------------------
ISO = COUNTRY_ISO3.upper()
INDEX_TILES = f"ookla-tiles-{ISO.lower()}"      # 1 document = 1 tuile Ookla (~610 m)
INDEX_ADMIN = f"ookla-admin-{ISO.lower()}"      # 1 document = 1 region x 1 reseau

ES_URL     = f"http://{ES_HOST}:{ES_PORT}"
KIBANA_URL = f"http://127.0.0.1:{KIBANA_PORT}"

# =============================================================================
#  CHARTE GRAPHIQUE  "AfDB-inspired Institutional"
# =============================================================================
AFDB = {
    "green":  "#00A86A",  # Accent principal
    "deep":   "#00704A",  # Kickers, accent secondaire
    "forest": "#00553A",  # Debut de degrade, barre la plus foncee
    "gold":   "#F5C242",  # Accent chaud (heros, numeros de section)
    "ochre":  "#D49A00",  # Alertes / mises en avant sur fond blanc
    "teal":   "#0E7C86",  # 3e couleur categorielle
    "terra":  "#C4621D",  # 4e couleur categorielle
    "brick":  "#B83B2E",  # Risques, valeurs negatives
    "ink":    "#231F20",  # Texte principal
    "slate":  "#5E6964",  # Texte secondaire, axes
    "mist":   "#F4F7F5",  # Fond de carte (card)
    "mint":   "#E8F5EF",  # Fond de carte mise en avant
    "sage":   "#D5DED9",  # Bordures, separateurs
    "grid":   "#E1E7E4",  # Gridlines tres legeres
}
RAMP = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]
SECTION_COLORS = [AFDB["green"], AFDB["deep"], AFDB["teal"],
                  AFDB["ochre"], AFDB["terra"], AFDB["brick"]]

# Classes de debit utilisees partout (prefixees pour garantir l'ordre alphabetique)
SPEED_CLASSES = [
    (0,    10,   "1 · < 10 Mbps"),
    (10,   25,   "2 · 10–25 Mbps"),
    (25,   100,  "3 · 25–100 Mbps"),
    (100,  1e9,  "4 · ≥ 100 Mbps"),
]

DASHBOARD_TITLE = f"[{ISO}] Connectivite Ookla x Population — {N_QUARTERS} trimestres"
print(f"Pays : {ISO}   |   {N_QUARTERS} trimestres jusqu'a {YEAR} Q{QUARTER}"
      f"   |   Index : {INDEX_TILES}, {INDEX_ADMIN}")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">01 &middot; ENVIRONNEMENT</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">D&eacute;tection de la plateforme et d&eacute;pendances</div>
</div>

Le notebook s'adapte tout seul&nbsp;:

| Plateforme | Cluster | Acc&egrave;s Kibana |
|---|---|---|
| **Google Colab** | Tarball install&eacute; dans la VM | Proxy de port Colab (URL publique automatique) |
| **Kaggle** | Tarball install&eacute; dans la VM (*Internet* doit &ecirc;tre activ&eacute;*) | Tunnel `cloudflared` optionnel, ou export NDJSON |
| **Local / WSL** | Tarball ou cluster existant | `http://localhost:5601` |
| **Docker** | Renseignez `ES_URL` / `KIBANA_URL` et mettez `INSTALL_STACK = False` | Votre URL |

In [ ]:
import os, sys, json, time, math, shutil, subprocess, textwrap, platform, getpass
from pathlib import Path

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working") and not IN_COLAB
PLATFORM  = "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else f"Local ({platform.system()})")

WORKDIR = Path("/content/work" if IN_COLAB else ("/kaggle/working/work" if IN_KAGGLE else "./ookla_work")).resolve()
DATADIR = WORKDIR / "data"; STACKDIR = WORKDIR / "stack"; OUTDIR = WORKDIR / "out"
for d in (DATADIR, STACKDIR, OUTDIR):
    d.mkdir(parents=True, exist_ok=True)

IS_LINUX = platform.system() == "Linux"
try:
    IS_ROOT = (os.geteuid() == 0)
except AttributeError:      # Windows
    IS_ROOT = False

print(f"Plateforme      : {PLATFORM}")
print(f"Python          : {sys.version.split()[0]}")
print(f"Repertoire      : {WORKDIR}")
print(f"Linux / root    : {IS_LINUX} / {IS_ROOT}")
if INSTALL_STACK and not IS_LINUX:
    print("\n[!] L'installation par tarball est reservee a Linux.")
    print("    Sur macOS/Windows : lancez Elastic via Docker puis mettez INSTALL_STACK = False.")

In [ ]:
# -----------------------------------------------------------------------------
#  Installation des dependances Python (uniquement celles qui manquent)
# -----------------------------------------------------------------------------
REQUIRED = [
    ("elasticsearch>=8.10,<9", "elasticsearch"),
    ("duckdb>=0.10",           "duckdb"),
    ("pandas",                 "pandas"),
    ("numpy",                  "numpy"),
    ("pyarrow",                "pyarrow"),
    ("requests",               "requests"),
    ("matplotlib",             "matplotlib"),
    ("shapely>=2.0",           "shapely"),
    ("geopandas>=0.14",        "geopandas"),
    ("rasterio>=1.3",          "rasterio"),
    ("tqdm",                   "tqdm"),
]

import importlib
missing = []
for spec, mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(spec)

if missing:
    print("Installation de :", ", ".join(missing))
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *missing]
    subprocess.run(cmd, check=False)
else:
    print("Toutes les dependances sont deja presentes.")

import numpy as np, pandas as pd, requests
import matplotlib as mpl, matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown
from tqdm.auto import tqdm

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("Imports OK —", ", ".join(f"{m}={importlib.import_module(m).__version__}" for m in ("pandas","numpy")))

In [ ]:
# -----------------------------------------------------------------------------
#  Helpers de presentation : style matplotlib + cartouches HTML (charte AfDB)
# -----------------------------------------------------------------------------
def afdb_style():
    mpl.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.sans-serif": ["Calibri", "Carlito", "DejaVu Sans"],
        "font.size": 10.5,
        "text.color": AFDB["ink"], "axes.labelcolor": AFDB["slate"],
        "axes.edgecolor": AFDB["sage"], "axes.linewidth": 0.8,
        "xtick.color": AFDB["slate"], "ytick.color": AFDB["slate"],
        "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
        "axes.grid": True, "grid.color": AFDB["grid"], "grid.linewidth": 0.7,
        "axes.axisbelow": True,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlepad": 14,
        "figure.dpi": 110, "legend.frameon": False,
    })
afdb_style()

def chart(ax, kicker, title, source=None):
    """Applique le gabarit AfDB a un axe : kicker en petites capitales + source en italique."""
    ax.set_title("")
    ax.text(0, 1.14, kicker.upper(), transform=ax.transAxes, fontsize=9,
            fontweight="bold", color=AFDB["deep"])
    ax.text(0, 1.045, title, transform=ax.transAxes, fontsize=13.5,
            fontweight="bold", color=AFDB["ink"])
    if source:
        ax.text(0, -0.17, source, transform=ax.transAxes, fontsize=8.5,
                style="italic", color=AFDB["slate"])
    return ax

def card(title, body, tone="neutral"):
    """Affiche un cartouche HTML (neutre / succes / alerte)."""
    fill, border, label = {
        "neutral": (AFDB["mist"], AFDB["sage"],  AFDB["deep"]),
        "ok":      (AFDB["mint"], AFDB["green"], AFDB["deep"]),
        "warn":    ("#FDF4E0",    AFDB["ochre"], AFDB["ochre"]),
        "risk":    ("#FBEDEB",    AFDB["brick"], AFDB["brick"]),
    }[tone]
    display(HTML(f"""
    <div style="background:{fill};border:1.4px solid {border};border-radius:12px;
                padding:14px 18px;margin:8px 0;font-family:Calibri,'Segoe UI',sans-serif;">
      <div style="font-size:10.5px;font-weight:700;letter-spacing:2.5px;text-transform:uppercase;color:{label};">{title}</div>
      <div style="font-size:14px;color:{AFDB['ink']};margin-top:6px;line-height:1.5;">{body}</div>
    </div>"""))

def kpi_row(items):
    """items = [(valeur, libelle, couleur), ...]"""
    cells = "".join(f"""
      <div style="flex:1;min-width:150px;background:#fff;border:1px solid {AFDB['sage']};
                  border-left:5px solid {c};border-radius:12px;padding:16px 18px;">
        <div style="font-size:30px;font-weight:700;color:{c};line-height:1;">{v}</div>
        <div style="font-size:11.5px;color:{AFDB['slate']};margin-top:8px;">{lbl}</div>
      </div>""" for v, lbl, c in items)
    display(HTML(f"<div style=\"display:flex;gap:12px;flex-wrap:wrap;font-family:Calibri,'Segoe UI',sans-serif;margin:10px 0;\">{cells}</div>"))

def fmt(n, unit=""):
    if n is None or (isinstance(n, float) and math.isnan(n)): return "—"
    if abs(n) >= 1e9: return f"{n/1e9:,.1f} Md{unit}"
    if abs(n) >= 1e6: return f"{n/1e6:,.1f} M{unit}"
    if abs(n) >= 1e3: return f"{n/1e3:,.1f} k{unit}"
    return f"{n:,.1f}{unit}"

card("Charte appliquee",
     "Palette, typographie et gabarits de graphiques align&eacute;s sur le guide de style "
     "<b>AfDB-inspired Institutional</b> &mdash; vert AfDB <code>#00A86A</code>, vert profond "
     "<code>#00704A</code>, or <code>#F5C242</code>.", tone="ok")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">02 &middot; CLUSTER</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Provisionner Elasticsearch et Kibana</div>
</div>

Nous montons un cluster **mono-n&oelig;ud** avec la s&eacute;curit&eacute; d&eacute;sactiv&eacute;e&nbsp;: c'est le mode *atelier*, jamais le mode production.

**Choix de configuration et pourquoi&nbsp;:**

| R&eacute;glage | Valeur | Raison |
|---|---|---|
| `discovery.type` | `single-node` | Pas d'&eacute;lection de master, d&eacute;marrage imm&eacute;diat |
| `xpack.security.enabled` | `false` | Pas de TLS/mot de passe &agrave; g&eacute;rer en salle de formation |
| `ES_JAVA_OPTS` | `-Xms1g -Xmx1g` | Colab/Kaggle offrent ~13 Go de RAM&nbsp;; 1 Go de *heap* suffit |
| `ingest.geoip.downloader.enabled` | `false` | &Eacute;vite un t&eacute;l&eacute;chargement inutile au d&eacute;marrage |
| `xpack.ml.enabled` | `false` | Lib&egrave;re de la m&eacute;moire |
| `network.host` | `127.0.0.1` | **Critique.** Une interface non-loopback fait basculer Elasticsearch en *mode production* et d&eacute;clenche les *bootstrap checks* (`vm.max_map_count` &agrave; 262144&hellip;), impossibles &agrave; satisfaire dans un conteneur Colab |
| `-Des.cgroups.hierarchy.override=/` | option JVM | **Critique.** Les conteneurs montent les statistiques cgroup &agrave; la racine en laissant les chemins inchang&eacute;s&nbsp;: sans cette propri&eacute;t&eacute;, `OsProbe` l&egrave;ve une `AccessControlException` et le n&oelig;ud meurt au d&eacute;marrage. L'image Docker officielle d'Elastic fait exactement la m&ecirc;me chose |

> **Si le n&oelig;ud refuse de d&eacute;marrer** &mdash; la fonction `diagnose()` extrait du log les lignes contenant `ERROR`, `Caused by`, `access denied` ou `bootstrap check`, au lieu d'afficher une queue tronqu&eacute;e. Les versions **9.x** d'Elasticsearch remplacent le `SecurityManager` Java par le syst&egrave;me d'*entitlements* et ne peuvent plus lever d'`AccessControlException`&nbsp;: si vous rencontrez encore ce type d'erreur, passer `ES_VERSION` en 9.x est une alternative.

> **Et le &laquo;&nbsp;L&nbsp;&raquo; d'ELK&nbsp;?** Logstash n'est pas n&eacute;cessaire ici&nbsp;: l'ingestion passe par l'API `_bulk` depuis Python, ce qui est plus simple &agrave; observer et &agrave; d&eacute;boguer en atelier. Elasticsearch refuse par ailleurs de d&eacute;marrer en tant que `root`&nbsp;: le notebook cr&eacute;e donc un utilisateur d&eacute;di&eacute; `esuser` quand il tourne en root (Colab/Kaggle).

In [ ]:
# -----------------------------------------------------------------------------
#  Utilitaires systeme : telechargement avec barre de progression + exec shell
# -----------------------------------------------------------------------------
RUN_USER = "esuser" if (IS_ROOT and IS_LINUX) else None

def sh(cmd, as_user=None, check=False, capture=False):
    """Execute une commande shell, eventuellement sous un autre utilisateur."""
    if as_user:
        cmd = f"su {as_user} -s /bin/bash -c {json.dumps(cmd)}"
    return subprocess.run(cmd, shell=True, check=check,
                          stdout=subprocess.PIPE if capture else None,
                          stderr=subprocess.STDOUT if capture else None,
                          text=True)

def download(url, dest, desc=None):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  (deja present) {dest.name}  —  {dest.stat().st_size/1e6:,.1f} Mo")
        return dest
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True,
                                         desc=desc or dest.name, leave=False) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk); bar.update(len(chunk))
    return dest

def service_up(url, path="/", timeout=2):
    try:
        return requests.get(url.rstrip("/") + path, timeout=timeout).status_code < 500
    except Exception:
        return False

def wait_for(check_fn, label, max_wait=300, every=5):
    t0 = time.time()
    with tqdm(total=max_wait, desc=f"Demarrage {label}", unit="s", leave=False) as bar:
        while time.time() - t0 < max_wait:
            if check_fn():
                bar.update(max_wait - bar.n)
                print(f"  {label} operationnel en {time.time()-t0:,.0f} s")
                return True
            time.sleep(every); bar.update(every)
    print(f"  [!] {label} n'a pas demarre en {max_wait} s")
    return False

print("Utilitaires systeme prets.")

In [ ]:
# -----------------------------------------------------------------------------
#  Telechargement + configuration d'Elasticsearch et Kibana
# -----------------------------------------------------------------------------
ES_HOME = STACKDIR / f"elasticsearch-{ES_VERSION}"
KB_HOME = STACKDIR / f"kibana-{ES_VERSION}"
ARTIFACTS = "https://artifacts.elastic.co/downloads"

# network.host reste en LOOPBACK : des qu'Elasticsearch ecoute sur une interface
# non-loopback il bascule en "mode production" et active les bootstrap checks
# (vm.max_map_count, descripteurs de fichiers...), impossibles a satisfaire dans un
# conteneur Colab/Kaggle. Seul Kibana a besoin d'etre joignable de l'exterieur.
ES_YML = textwrap.dedent(f"""
    cluster.name: ookla-workshop
    node.name: node-1
    network.host: 127.0.0.1
    http.port: {ES_PORT}
    discovery.type: single-node
    xpack.security.enabled: false
    xpack.security.enrollment.enabled: false
    xpack.ml.enabled: false
    ingest.geoip.downloader.enabled: false
    bootstrap.memory_lock: false
    action.destructive_requires_name: false
""").strip() + "\n"

# Colab/Kaggle sont des conteneurs : le montage des cgroups y est remanie, ce qui fait
# echouer OsProbe.getCgroup() au demarrage (AccessControlException). Elastic fournit
# la propriete es.cgroups.hierarchy.override pour ce cas precis — c'est ce que fait
# l'entrypoint de l'image Docker officielle. La politique additionnelle est une
# ceinture de securite sur la lecture des pseudo-systemes de fichiers.
ES_EXTRA_POLICY = (
    "grant {\n"
    '  permission java.io.FilePermission "/sys/fs/cgroup", "read";\n'
    '  permission java.io.FilePermission "/sys/fs/cgroup/-", "read";\n'
    '  permission java.io.FilePermission "/proc/-", "read";\n'
    "};\n"
)

KEY = "afdb_ookla_workshop_encryption_key_32+"
KB_YML = textwrap.dedent(f"""
    server.host: "0.0.0.0"
    server.port: {KIBANA_PORT}
    server.publicBaseUrl: "http://localhost:{KIBANA_PORT}"
    elasticsearch.hosts: ["http://127.0.0.1:{ES_PORT}"]
    telemetry.optIn: false
    telemetry.enabled: false
    xpack.encryptedSavedObjects.encryptionKey: "{KEY}"
    xpack.reporting.encryptionKey: "{KEY}"
    xpack.security.encryptionKey: "{KEY}"
    logging.root.level: warn
""").strip() + "\n"

def install_stack():
    if not IS_LINUX:
        raise RuntimeError("Installation par tarball uniquement sur Linux — utilisez Docker.")
    if IS_ROOT:
        sh(f"id -u {RUN_USER} >/dev/null 2>&1 || useradd -m {RUN_USER}", capture=True)

    if not ES_HOME.exists():
        print("[1/4] Telechargement d'Elasticsearch", ES_VERSION)
        tgz = download(f"{ARTIFACTS}/elasticsearch/elasticsearch-{ES_VERSION}-linux-x86_64.tar.gz",
                       STACKDIR / f"es-{ES_VERSION}.tar.gz", "elasticsearch")
        print("[2/4] Extraction..."); sh(f"tar -xzf {tgz} -C {STACKDIR}", check=True)
    if not KB_HOME.exists():
        print("[3/4] Telechargement de Kibana", ES_VERSION, "(~1 Go, patience)")
        tgz = download(f"{ARTIFACTS}/kibana/kibana-{ES_VERSION}-linux-x86_64.tar.gz",
                       STACKDIR / f"kb-{ES_VERSION}.tar.gz", "kibana")
        print("[4/4] Extraction...")
        sh(f"tar -xzf {tgz} -C {STACKDIR}", check=True)
        extracted = STACKDIR / f"kibana-{ES_VERSION}-linux-x86_64"
        if extracted.exists() and not KB_HOME.exists():
            extracted.rename(KB_HOME)

    cfg = ES_HOME / "config"
    (cfg / "elasticsearch.yml").write_text(ES_YML)
    (cfg / "extra.policy").write_text(ES_EXTRA_POLICY)
    (cfg / "jvm.options.d").mkdir(exist_ok=True)
    (cfg / "jvm.options.d" / "workshop.options").write_text(
        "-Des.cgroups.hierarchy.override=/\n"
        f"-Djava.security.policy={cfg / 'extra.policy'}\n"
        f"-Xms{ES_HEAP}\n-Xmx{ES_HEAP}\n")
    (KB_HOME / "config" / "kibana.yml").write_text(KB_YML)
    if IS_ROOT:
        sh("sysctl -w vm.max_map_count=262144", capture=True)   # sans effet si interdit
        sh(f"chown -R {RUN_USER}:{RUN_USER} {STACKDIR}", capture=True)
    print("Stack installee dans", STACKDIR)

if INSTALL_STACK and not service_up(ES_URL):
    install_stack()
elif service_up(ES_URL):
    print("Un cluster repond deja sur", ES_URL, "— installation ignoree.")
else:
    print("INSTALL_STACK = False : le notebook utilisera", ES_URL)

In [ ]:
# -----------------------------------------------------------------------------
#  Demarrage des services
# -----------------------------------------------------------------------------
KEYWORDS = ("ERROR", "FATAL", "Caused by", "access denied", "bootstrap check",
            "uncaught exception", "fatal exception")

def diagnose(path, label):
    """Affiche les lignes reellement utiles du log, pas une queue tronquee."""
    if not Path(path).exists():
        print(f"  Aucun log {label} trouve."); return
    txt = Path(path).read_text(errors="replace")
    hits = [l for l in txt.split("\n") if any(k in l for k in KEYWORDS)]
    print(f"\n--- Diagnostic {label} ---")
    for l in (hits[:14] if hits else txt.split("\n")[-14:]):
        print("  " + l[:280])

def start_es():
    if service_up(ES_URL):
        print("Elasticsearch deja actif."); return True
    # Le heap et les options JVM sont dans config/jvm.options.d/workshop.options :
    # on ne repasse pas par ES_JAVA_OPTS pour eviter les reglages en double.
    env = f"export ES_TMPDIR={STACKDIR}/tmp; mkdir -p {STACKDIR}/tmp; "
    sh(env + f"{ES_HOME}/bin/elasticsearch -d -p {STACKDIR}/es.pid", as_user=RUN_USER, capture=True)
    ok = wait_for(lambda: service_up(ES_URL), "Elasticsearch", max_wait=240)
    if not ok:
        logs = sorted((ES_HOME / "logs").glob("*.log"))
        diagnose(logs[-1] if logs else ES_HOME / "logs" / "absent.log", "Elasticsearch")
        print("\n  Pistes : verifier network.host (doit rester 127.0.0.1),")
        print("  la presence de config/jvm.options.d/workshop.options, et les droits sur", STACKDIR)
    return ok

def start_kibana():
    if service_up(KIBANA_URL, "/api/status"):
        print("Kibana deja actif."); return True
    logf = STACKDIR / "kibana.log"
    sh(f"nohup {KB_HOME}/bin/kibana > {logf} 2>&1 &", as_user=RUN_USER, capture=True)
    def ready():
        try:
            s = requests.get(KIBANA_URL + "/api/status", timeout=3).json()
            return s.get("status", {}).get("overall", {}).get("level") == "available"
        except Exception:
            return False
    ok = wait_for(ready, "Kibana", max_wait=420, every=8)
    if not ok:
        diagnose(logf, "Kibana")
    return ok

if INSTALL_STACK or not service_up(ES_URL):
    if start_es():
        start_kibana()

if not service_up(ES_URL):
    card("Cluster indisponible",
         "Elasticsearch n'a pas demarre. Lisez le diagnostic ci-dessus avant de continuer&nbsp;: "
         "les causes les plus frequentes sont un <code>network.host</code> non-loopback "
         "(bootstrap checks) et la detection des cgroups en conteneur.", tone="risk")
    raise SystemExit("Elasticsearch indisponible — corrigez avant de poursuivre.")

info = requests.get(ES_URL, timeout=10).json()
kpi_row([
    (info["version"]["number"],                "Version Elasticsearch",  AFDB["green"]),
    (info["cluster_name"],                     "Nom du cluster",         AFDB["deep"]),
    (requests.get(f"{ES_URL}/_cluster/health", timeout=10).json()["status"].upper(),
                                               "Sante du cluster",       AFDB["teal"]),
])

In [ ]:
# -----------------------------------------------------------------------------
#  Client Python + verification de la sante du cluster
# -----------------------------------------------------------------------------
from elasticsearch import Elasticsearch, helpers

es = Elasticsearch(ES_URL, request_timeout=180, retry_on_timeout=True, max_retries=3)
health = es.cluster.health()
print(json.dumps({k: health[k] for k in
      ("cluster_name","status","number_of_nodes","active_shards","unassigned_shards")}, indent=2))

# La version de la JVM vit dans _nodes/info, pas dans _nodes/stats (qui ne porte
# que des compteurs). nodes.info donne version ET taille du heap en un seul appel.
for n in es.nodes.info(metric="jvm")["nodes"].values():
    jvm = n["jvm"]
    print(f"Noeud {n['name']} — heap max {jvm['mem']['heap_max_in_bytes']/1e9:,.1f} Go, "
          f"JVM {jvm['version']} ({jvm.get('vm_name', '?')})")

# Statut "yellow" attendu sur un noeud unique : les index systeme de Kibana demandent
# une replique qui ne peut jamais etre allouee. Nos index sont crees en replicas: 0.
if health["status"] == "yellow":
    print(f"\nStatut yellow : {health['unassigned_shards']} shards non assignes "
          "(repliques d'index systeme). Sans consequence pour ce TP.")
    # Pour forcer le vert : es.indices.put_settings(index="*", expand_wildcards="all",
    #                          settings={"index.number_of_replicas": 0})

### Ouvrir Kibana maintenant

Kibana &eacute;coute sur le port 5601 **&agrave; l'int&eacute;rieur de la VM**, qui n'est pas expos&eacute;e sur Internet. Chaque plateforme a son m&eacute;canisme&nbsp;:

* **Colab** &mdash; `google.colab.kernel.proxyPort(5601)` g&eacute;n&egrave;re une URL `googleusercontent.com`. Elle est li&eacute;e &agrave; votre compte Google&nbsp;: ouvrez-la dans le **m&ecirc;me profil de navigateur**, sinon vous obtiendrez une erreur d'authentification.
* **Local** &mdash; `http://localhost:5601`, rien &agrave; faire.
* **Kaggle** &mdash; aucun port expos&eacute;&nbsp;: la cellule propose un tunnel `cloudflared` gratuit.

Le premier d&eacute;marrage de Kibana prend 1 &agrave; 3 minutes. La cellule attend que le statut passe &agrave; `available` avant de donner l'URL.

In [ ]:
# -----------------------------------------------------------------------------
#  Acces a l'interface Kibana
# -----------------------------------------------------------------------------
def kibana_status():
    try:
        return requests.get(KIBANA_URL + "/api/status", timeout=5) \
                       .json()["status"]["overall"]["level"]
    except Exception:
        return None

def kibana_public_url(open_window=True):
    """Renvoie une URL ouvrable depuis le navigateur, selon la plateforme."""
    lvl = kibana_status()
    if lvl != "available":
        print(f"Kibana n'est pas pret (statut : {lvl}).")
        print("  -> relancez start_kibana() puis reexecutez cette cellule.")
        print(f"  -> journal : !tail -n 30 {STACKDIR}/kibana.log")
        return None
    if IN_COLAB:
        from google.colab.output import eval_js, serve_kernel_port_as_window
        url = eval_js(f"google.colab.kernel.proxyPort({KIBANA_PORT})").rstrip("/")
        if open_window:
            try: serve_kernel_port_as_window(KIBANA_PORT)
            except Exception: pass
        return url
    if IN_KAGGLE:
        print("Kaggle n'expose aucun port. Tunnel gratuit :")
        print("  !wget -q https://github.com/cloudflare/cloudflared/releases/latest/"
              "download/cloudflared-linux-amd64 -O /usr/local/bin/cf && chmod +x /usr/local/bin/cf")
        print(f"  !nohup cf tunnel --url http://localhost:{KIBANA_PORT} > /tmp/cf.log 2>&1 &")
        print("  !grep -o 'https://.*trycloudflare.com' /tmp/cf.log | head -1")
        return None
    return KIBANA_URL

KIBANA_PUBLIC = kibana_public_url()
if KIBANA_PUBLIC:
    display(HTML(f"""
    <div style="background:{AFDB['mint']};border:1.5px solid {AFDB['green']};border-radius:12px;
                padding:16px 20px;font-family:Calibri,'Segoe UI',sans-serif;">
      <div style="font-size:10.5px;font-weight:700;letter-spacing:2.5px;color:{AFDB['deep']};">KIBANA DISPONIBLE</div>
      <div style="font-size:14px;margin-top:8px;line-height:1.9;">
        <a href="{KIBANA_PUBLIC}" target="_blank" style="color:{AFDB['deep']};font-weight:700;">Accueil</a> &nbsp;·&nbsp;
        <a href="{KIBANA_PUBLIC}/app/discover" target="_blank" style="color:{AFDB['deep']};font-weight:700;">Discover</a> &nbsp;·&nbsp;
        <a href="{KIBANA_PUBLIC}/app/maps" target="_blank" style="color:{AFDB['deep']};font-weight:700;">Maps</a> &nbsp;·&nbsp;
        <a href="{KIBANA_PUBLIC}/app/dashboards" target="_blank" style="color:{AFDB['deep']};font-weight:700;">Dashboards</a>
      </div>
      <div style="font-size:12px;color:{AFDB['slate']};margin-top:8px;">
        Les data views n'existeront qu'apres la section 09. Pour l'instant, le cluster est vide.
      </div>
    </div>"""))
    print(KIBANA_PUBLIC)

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">03 &middot; DONN&Eacute;ES OOKLA</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Le quadkey, ou comment t&eacute;l&eacute;charger 0,3&nbsp;% d'un fichier de 2&nbsp;Go</div>
</div>

Ookla publie chaque trimestre, en *open data*, les performances agr&eacute;g&eacute;es par **tuile Web Mercator de zoom 16** (&asymp;&nbsp;610&nbsp;m &agrave; l'&eacute;quateur). Un fichier Parquet **mondial** par trimestre et par type de r&eacute;seau, soit plusieurs millions de lignes.

**Sch&eacute;ma source** &mdash; `quadkey`, `tile` (polygone WKT), `avg_d_kbps`, `avg_u_kbps`, `avg_lat_ms`, `tests`, `devices`.

### Pourquoi le quadkey change tout

Un *quadkey* est l'encodage en base&nbsp;4 du chemin d'une tuile dans le quadtree Web Mercator. Deux propri&eacute;t&eacute;s en d&eacute;coulent&nbsp;:

1. **Pr&eacute;fixe = contenance spatiale.** La tuile `1202` de zoom&nbsp;4 contient toutes les tuiles de zoom&nbsp;16 dont le quadkey commence par `1202`. Filtrer une zone revient donc &agrave; filtrer des **pr&eacute;fixes de cha&icirc;nes**.
2. **Ordre lexicographique &asymp; ordre spatial** (courbe en Z). Le fichier Parquet &eacute;tant tri&eacute; par quadkey, les statistiques min/max de chaque *row group* permettent &agrave; DuckDB d'&eacute;liminer &agrave; distance la quasi-totalit&eacute; des blocs&nbsp;: c'est du **predicate pushdown** sur HTTP Range requests.

R&eacute;sultat&nbsp;: quelques dizaines de m&eacute;gaoctets t&eacute;l&eacute;charg&eacute;s au lieu de deux gigaoctets, sans serveur interm&eacute;diaire.

In [ ]:
# -----------------------------------------------------------------------------
#  Mathematiques du quadkey (Web Mercator / quadtree Bing Maps)
# -----------------------------------------------------------------------------
Z_OOKLA = 16   # zoom des tuiles Ookla

def lonlat_to_tile_xy(lon, lat, z):
    """(lon, lat) degres -> indices de tuile (x, y) au zoom z."""
    lat = max(min(lat, 85.05112878), -85.05112878)
    n = 2 ** z
    x = int((lon + 180.0) / 360.0 * n)
    s = math.sin(math.radians(lat))
    y = int((0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)) * n)
    return min(max(x, 0), n - 1), min(max(y, 0), n - 1)

def tile_xy_to_quadkey(x, y, z):
    """Indices de tuile -> quadkey (chaine de z chiffres en base 4)."""
    out = []
    for i in range(z, 0, -1):
        digit, mask = 0, 1 << (i - 1)
        if x & mask: digit += 1
        if y & mask: digit += 2
        out.append(str(digit))
    return "".join(out)

def quadkey_prefixes_for_bbox(bbox, max_prefixes=96):
    """Plus petit ensemble de prefixes couvrant la bbox (minx, miny, maxx, maxy)."""
    minx, miny, maxx, maxy = bbox
    for z in range(10, 1, -1):
        x0, y0 = lonlat_to_tile_xy(minx, maxy, z)
        x1, y1 = lonlat_to_tile_xy(maxx, miny, z)
        n = (x1 - x0 + 1) * (y1 - y0 + 1)
        if n <= max_prefixes:
            return [tile_xy_to_quadkey(x, y, z)
                    for x in range(x0, x1 + 1) for y in range(y0, y1 + 1)], z, n
    raise RuntimeError("Bbox trop large")

def _quadkey_succ(s):
    """Successeur immediat d'un quadkey dans l'ordre lexicographique (base 4)."""
    d = list(s)
    for i in range(len(d) - 1, -1, -1):
        if d[i] != "3":
            d[i] = str(int(d[i]) + 1)
            return "".join(d[:i + 1]) + "0" * (len(d) - i - 1)
        d[i] = "0"
    return None

def quadkey_ranges(prefixes, z_target=Z_OOKLA):
    """Prefixes -> intervalles [lo, hi] fusionnes, exploitables par un BETWEEN SQL.

    Les prefixes contigus sur la courbe en Z sont aussi contigus lexicographiquement :
    les fusionner reduit fortement le nombre de clauses OR (ex. Tunisie : 91 -> 23).
    """
    rng = sorted((p + "0" * (z_target - len(p)), p + "3" * (z_target - len(p)))
                 for p in prefixes)
    merged = []
    for lo, hi in rng:
        if merged and (lo <= merged[-1][1] or lo == _quadkey_succ(merged[-1][1])):
            merged[-1][1] = max(merged[-1][1], hi)
        else:
            merged.append([lo, hi])
    return merged

def quadkeys_to_bounds(qk_series):
    """Vectorise : Series de quadkeys -> (min_lon, min_lat, max_lon, max_lat) en degres."""
    arr = np.asarray(qk_series, dtype=str)
    z = len(arr[0])
    digits = (np.frombuffer("".join(arr.tolist()).encode("ascii"), dtype=np.uint8)
                .reshape(-1, z).astype(np.int64) - 48)
    weights = (1 << np.arange(z - 1, -1, -1)).astype(np.int64)
    x = ((digits & 1) * weights).sum(axis=1)
    y = (((digits >> 1) & 1) * weights).sum(axis=1)
    n = float(1 << z)
    lon0 = x / n * 360.0 - 180.0
    lon1 = (x + 1) / n * 360.0 - 180.0
    lat0 = np.degrees(np.arctan(np.sinh(np.pi * (1 - 2 * y / n))))
    lat1 = np.degrees(np.arctan(np.sinh(np.pi * (1 - 2 * (y + 1) / n))))
    return lon0, np.minimum(lat0, lat1), lon1, np.maximum(lat0, lat1)

# Demonstration pedagogique
demo_qk = tile_xy_to_quadkey(*lonlat_to_tile_xy(10.18, 36.80, Z_OOKLA), Z_OOKLA)  # Tunis
b = quadkeys_to_bounds(pd.Series([demo_qk]))
card("Verification",
     f"Tunis (10.18, 36.80) &rarr; quadkey <code>{demo_qk}</code> au zoom 16.<br>"
     f"Emprise reconstruite : lon [{b[0][0]:.5f}, {b[2][0]:.5f}], lat [{b[1][0]:.5f}, {b[3][0]:.5f}]<br>"
     f"Prefixe de zoom 6 : <code>{demo_qk[:6]}</code> &mdash; il contient {4**10:,} tuiles de zoom 16.")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">04 &middot; EMPRISE & T&Eacute;L&Eacute;CHARGEMENT</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Fronti&egrave;res geoBoundaries, puis s&eacute;rie trimestrielle</div>
</div>

**geoBoundaries** (William & Mary, CC BY) fournit ADM0 et ADM1. On s'en sert pour l'**emprise** (construction des pr&eacute;fixes quadkey) et pour la **d&eacute;coupe exacte** avec rattachement r&eacute;gional.

Nouveaut&eacute; par rapport &agrave; une extraction ponctuelle&nbsp;: on t&eacute;l&eacute;charge **`N_QUARTERS` trimestres cons&eacute;cutifs**. C'est ce qui donne un sens au s&eacute;lecteur de dates de Kibana &mdash; avec un seul trimestre, tous les documents portent la m&ecirc;me date et le filtre temporel ne filtre rien. Le co&ucirc;t est proportionnel&nbsp;: quatre trimestres, c'est quatre requ&ecirc;tes DuckDB et quatre fois plus de documents.

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union

GB_API = "https://www.geoboundaries.org/api/current/gbOpen/{iso}/{lvl}/"

def load_boundaries(iso3, level):
    """Telecharge un niveau administratif depuis geoBoundaries (GeoJSON, EPSG:4326)."""
    meta = requests.get(GB_API.format(iso=iso3.upper(), lvl=level), timeout=90).json()
    if isinstance(meta, list): meta = meta[0]
    url = meta.get("gjDownloadURL") or meta.get("staticDownloadLink")
    gdf = gpd.read_file(url)
    if gdf.crs is None: gdf.set_crs(4326, inplace=True)
    return gdf.to_crs(4326), meta

adm0, meta0 = load_boundaries(ISO, "ADM0")
COUNTRY_NAME = meta0.get("boundaryName", ISO)

try:
    adm1, _ = load_boundaries(ISO, "ADM1")
    adm1 = adm1.rename(columns={"shapeName": "admin1"})[["admin1", "geometry"]]
    HAS_ADM1 = True
except Exception as exc:
    print("ADM1 indisponible :", exc)
    adm1 = adm0.assign(admin1=COUNTRY_NAME)[["admin1", "geometry"]]
    HAS_ADM1 = False

# Nettoyage geometrique (auto-intersections frequentes dans les donnees ouvertes)
adm0["geometry"] = adm0.geometry.buffer(0)
adm1["geometry"] = adm1.geometry.buffer(0)
COUNTRY_GEOM = unary_union(adm0.geometry.values)
BBOX = tuple(adm0.total_bounds)

prefixes, z_pref, n_pref = quadkey_prefixes_for_bbox(BBOX)
RANGES = quadkey_ranges(prefixes)

kpi_row([
    (COUNTRY_NAME,            "Pays",                         AFDB["green"]),
    (f"{len(adm1)}",          "Regions ADM1" if HAS_ADM1 else "ADM0 seulement", AFDB["deep"]),
    (f"{n_pref} / z{z_pref}", "Prefixes quadkey",             AFDB["teal"]),
    (f"{len(RANGES)}",        "Intervalles SQL apres fusion", AFDB["ochre"]),
])
print(f"Bounding box : lon [{BBOX[0]:.3f}, {BBOX[2]:.3f}]  lat [{BBOX[1]:.3f}, {BBOX[3]:.3f}]")

In [ ]:
# -----------------------------------------------------------------------------
#  Extraction distante du Parquet Ookla, trimestre par trimestre
# -----------------------------------------------------------------------------
import duckdb

QUARTER_MONTH = {1: "01", 2: "04", 3: "07", 4: "10"}

def quarters_back(year, quarter, n):
    """[(annee, trimestre)] des n trimestres finissant sur (year, quarter), ordre croissant."""
    out, y, q = [], year, quarter
    for _ in range(n):
        out.append((y, q))
        q -= 1
        if q == 0: y, q = y - 1, 4
    return list(reversed(out))

PERIODS = quarters_back(YEAR, QUARTER, N_QUARTERS)
print("Trimestres vises :", ", ".join(f"{y} Q{q}" for y, q in PERIODS))

def ookla_url(network, year, quarter):
    return (f"https://ookla-open-data.s3.amazonaws.com/parquet/performance/"
            f"type={network}/year={year}/quarter={quarter}/"
            f"{year}-{QUARTER_MONTH[quarter]}-01_performance_{network}_tiles.parquet")

def fetch_ookla(network, year, quarter, ranges, limit=None):
    con = duckdb.connect()
    for stmt in ("INSTALL httpfs;", "LOAD httpfs;"):
        try: con.execute(stmt)
        except Exception: pass
    where = " OR ".join(f"(quadkey BETWEEN '{lo}' AND '{hi}')" for lo, hi in ranges)
    sql = f"""SELECT quadkey, avg_d_kbps, avg_u_kbps, avg_lat_ms, tests, devices
              FROM read_parquet('{ookla_url(network, year, quarter)}')
              WHERE {where} {f'LIMIT {limit}' if limit else ''}"""
    df = con.execute(sql).df()
    con.close()
    return df

raw = {}
for (y, q) in PERIODS:
    for net in NETWORK_TYPES:
        t0 = time.time()
        try:
            df = fetch_ookla(net, y, q, RANGES, limit=MAX_TILES_PER_PERIOD)
            raw[(net, y, q)] = df
            print(f"  {y} Q{q} · {net:<6} : {len(df):>8,} tuiles  ({time.time()-t0:,.1f} s)")
        except Exception as exc:
            raw[(net, y, q)] = pd.DataFrame()
            print(f"  {y} Q{q} · {net:<6} : ECHEC — {str(exc)[:110]}")

got = sorted({(y, q) for (n, y, q), v in raw.items() if len(v)})
assert got, "Aucune donnee Ookla recuperee — verifiez YEAR/QUARTER, N_QUARTERS et l'acces Internet."

lost = [f"{y} Q{q}" for (y, q) in PERIODS if (y, q) not in got]
kpi_row([
    (f"{len(got)}/{len(PERIODS)}",             "Trimestres recuperes", AFDB["green"]),
    (f"{sum(len(v) for v in raw.values()):,}", "Tuiles brutes",        AFDB["deep"]),
    (f"{len({n for (n, _, _), v in raw.items() if len(v)})}", "Types de reseau", AFDB["teal"]),
])
if lost:
    card("Trimestres manquants",
         f"Aucune donnee pour&nbsp;: <b>{', '.join(lost)}</b>. Ookla publie avec un d&eacute;calage "
         "de quelques semaines&nbsp;; un trimestre trop r&eacute;cent n'existe pas encore. "
         "Reculez <code>QUARTER</code> d'un cran et relancez depuis les param&egrave;tres.", tone="warn")

In [ ]:
# -----------------------------------------------------------------------------
#  Geometrie des tuiles, decoupe pays, rattachement ADM1
# -----------------------------------------------------------------------------
def enrich(df, network, year, quarter):
    if df.empty: return df
    df = df.copy()
    lon0, lat0, lon1, lat1 = quadkeys_to_bounds(df["quadkey"])
    df["min_lon"], df["min_lat"], df["max_lon"], df["max_lat"] = lon0, lat0, lon1, lat1
    df["lon"], df["lat"] = (lon0 + lon1) / 2.0, (lat0 + lat1) / 2.0
    df["tile_km2"] = ((lon1 - lon0) * 111.320 * np.cos(np.radians(df["lat"]))) * \
                     ((lat1 - lat0) * 110.574)
    df["download_mbps"] = df["avg_d_kbps"] / 1000.0
    df["upload_mbps"]   = df["avg_u_kbps"] / 1000.0
    df["latency_ms"]    = df["avg_lat_ms"].astype(float)
    df["network"]       = network
    df["period"]        = f"{year}-{QUARTER_MONTH[quarter]}-01"
    df["year_quarter"]  = f"{year} Q{quarter}"   # cle categorielle : ordre alpha = ordre chrono
    return df

tiles = pd.concat([enrich(df, n, y, q) for (n, y, q), df in raw.items() if len(df)],
                  ignore_index=True)
print(f"Tuiles avant decoupe : {len(tiles):,}")

pts = gpd.GeoDataFrame(tiles[["quadkey"]].copy(),
                       geometry=gpd.points_from_xy(tiles["lon"], tiles["lat"]), crs=4326)
joined = gpd.sjoin(pts, adm1[["admin1", "geometry"]], how="left", predicate="within")
joined = joined[~joined.index.duplicated(keep="first")].reindex(pts.index)
tiles["admin1"] = joined["admin1"].values

before = len(tiles)
tiles = tiles[tiles["admin1"].notna()].reset_index(drop=True)
print(f"Tuiles retenues dans {COUNTRY_NAME} : {len(tiles):,}  "
      f"({before - len(tiles):,} ecartees : mer, pays voisins)")

def speed_class(v):
    for lo, hi, label in SPEED_CLASSES:
        if lo <= v < hi: return label
    return SPEED_CLASSES[-1][2]

tiles["speed_class"] = tiles["download_mbps"].apply(speed_class)
QUARTER_LABELS = sorted(tiles["year_quarter"].unique())
LATEST_Q = QUARTER_LABELS[-1]
display(tiles.head(4)[["quadkey","year_quarter","network","admin1","download_mbps",
                       "latency_ms","tests","speed_class"]])

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">05 &middot; POPULATION</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Croiser la connectivit&eacute; avec WorldPop</div>
</div>

Une tuile Ookla dit &agrave; quelle vitesse on se connecte, **pas combien de personnes sont concern&eacute;es**. WorldPop (*Global 1 km, UN-adjusted*) fournit une grille o&ugrave; chaque pixel porte un nombre d'habitants.

Deux calculs distincts&nbsp;:

* **Population par tuile** &mdash; valeur du pixel sous le centre de la tuile, convertie en **densit&eacute; (hab/km&sup2;)** en divisant par la surface r&eacute;elle du pixel &agrave; cette latitude, puis multipli&eacute;e par la surface de la tuile. C'est une **estimation**.
* **Population par r&eacute;gion** &mdash; statistique zonale (somme des pixels dans le polygone). C'est le d&eacute;nominateur du taux de couverture.

La population est suppos&eacute;e constante sur la p&eacute;riode&nbsp;: WorldPop ne publie ce produit qu'annuellement, et l'&eacute;volution sur quelques trimestres est n&eacute;gligeable devant l'incertitude du mod&egrave;le.

In [ ]:
import rasterio
from rasterio.mask import mask as rio_mask

def worldpop_url(iso3, year):
    return (f"https://data.worldpop.org/GIS/Population/Global_2000_2020_1km_UNadj/"
            f"{year}/{iso3.upper()}/{iso3.lower()}_ppp_{year}_1km_Aggregated_UNadj.tif")

POP_TIF = DATADIR / f"worldpop_{ISO.lower()}_{WORLDPOP_YEAR}.tif"
HAS_POP = True
try:
    download(worldpop_url(ISO, WORLDPOP_YEAR), POP_TIF, "worldpop")
    with rasterio.open(POP_TIF) as src:
        print(f"Raster : {src.width} x {src.height} px, resolution {src.res[0]:.5f} deg, "
              f"CRS {src.crs}, nodata {src.nodata}")
except Exception as exc:
    HAS_POP = False
    print("[!] WorldPop indisponible :", exc)

In [ ]:
# -----------------------------------------------------------------------------
#  Population estimee par tuile (lecture vectorisee du raster)
# -----------------------------------------------------------------------------
def attach_population(df, tif):
    with rasterio.open(tif) as src:
        arr = src.read(1).astype("float64")
        if src.nodata is not None: arr[arr == src.nodata] = np.nan
        arr[arr < 0] = np.nan
        rows, cols = rasterio.transform.rowcol(src.transform,
                                               df["lon"].values, df["lat"].values)
        rows, cols = np.asarray(rows), np.asarray(cols)
        ok = (rows >= 0) & (rows < arr.shape[0]) & (cols >= 0) & (cols < arr.shape[1])
        vals = np.full(len(df), np.nan)
        vals[ok] = arr[rows[ok], cols[ok]]
        res_x, res_y = src.res
    px_km2 = (res_x * 111.320 * np.cos(np.radians(df["lat"].values))) * (res_y * 110.574)
    density = vals / px_km2
    return np.nan_to_num(density * df["tile_km2"].values, nan=0.0), np.nan_to_num(density, nan=0.0)

if HAS_POP:
    tiles["population"], tiles["pop_density"] = attach_population(tiles, POP_TIF)
else:
    tiles["population"], tiles["pop_density"] = 0.0, 0.0

# Prefixes numeriques : garantissent l'ordre des categories dans Kibana
tiles["settlement"] = np.where(tiles["pop_density"] >= 1500, "3 · Urbain dense",
                       np.where(tiles["pop_density"] >= 300, "2 · Urbain", "1 · Rural"))

latest = tiles[tiles["year_quarter"] == LATEST_Q]
kpi_row([
    (f"{len(tiles):,}",                          "Tuiles, tous trimestres",        AFDB["green"]),
    (fmt(tiles["tests"].sum()),                  "Tests de debit cumules",         AFDB["deep"]),
    (fmt(latest["population"].sum()),            "Population sous tuile mesuree",  AFDB["teal"]),
    (f"{latest['download_mbps'].median():,.1f}", f"Debit median, {LATEST_Q}",      AFDB["ochre"]),
])

In [ ]:
# -----------------------------------------------------------------------------
#  Population totale par region (statistique zonale)
# -----------------------------------------------------------------------------
def zonal_population(gdf, tif):
    out = {}
    with rasterio.open(tif) as src:
        for _, row in tqdm(list(gdf.iterrows()), desc="Statistiques zonales", leave=False):
            try:
                data, _ = rio_mask(src, [row.geometry.__geo_interface__], crop=True, filled=True)
                a = data[0].astype("float64")
                if src.nodata is not None: a[a == src.nodata] = np.nan
                a[a < 0] = np.nan
                out[row["admin1"]] = float(np.nansum(a))
            except Exception:
                out[row["admin1"]] = float("nan")
    return out

ADMIN_POP = zonal_population(adm1, POP_TIF) if HAS_POP else {a: float("nan") for a in adm1["admin1"]}
pop_total = np.nansum(list(ADMIN_POP.values()))
print(f"Population totale {COUNTRY_NAME} ({WORLDPOP_YEAR}) : {pop_total:,.0f} habitants")
display(pd.Series(ADMIN_POP, name="population").sort_values(ascending=False).head(8).to_frame())

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">06 &middot; MAPPING</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Le mapping, et pourquoi <code>geo_shape</code></div>
</div>

Le *mapping* est le sch&eacute;ma de l'index&nbsp;: il fige le type de chaque champ **avant** l'indexation. Un d&eacute;bit devenu `text` rend toute agr&eacute;gation num&eacute;rique impossible sans r&eacute;indexation compl&egrave;te.

### `geo_point` vs `geo_shape`

| | `geo_point` | `geo_shape` |
|---|---|---|
| Repr&eacute;sente | un couple lat/lon | point, ligne, polygone, **envelope**, multi-g&eacute;om&eacute;tries |
| Structure interne | BKD-tree 2D | BKD-tree sur les **cellules** d&eacute;coup&eacute;es de la g&eacute;om&eacute;trie |
| Requ&ecirc;tes | `geo_distance`, `geo_bounding_box`, `geo_grid` | `geo_shape` (`intersects`, `within`, `contains`, `disjoint`) |
| Agr&eacute;gations | `geo_centroid`, `geohash_grid`, `geotile_grid`, `geo_bounds` | `geotile_grid`, `geo_bounds` |
| Co&ucirc;t | faible | plus &eacute;lev&eacute; (indexation & stockage) |

**Notre choix&nbsp;: les deux.** `tile_geom` en `geo_shape` de type **`envelope`** (deux coins suffisent pour un rectangle align&eacute; sur les axes&nbsp;: la forme la plus compacte pour une tuile Mercator) et `centroid` en `geo_point` (indispensable pour `geo_distance` et les agr&eacute;gations de grille).

Le champ **`period`** est un `date`&nbsp;: c'est lui qui alimentera le s&eacute;lecteur temporel de Kibana. Le champ `year_quarter` est son pendant `keyword`, plus commode pour un axe cat&eacute;goriel.

In [ ]:
# -----------------------------------------------------------------------------
#  Mapping de l'index des tuiles
# -----------------------------------------------------------------------------
TILES_SETTINGS = {"number_of_shards": 1, "number_of_replicas": 0, "refresh_interval": "30s"}

TILES_MAPPINGS = {
    "dynamic": "strict",           # tout champ non declare est rejete : garde-fou pedagogique
    "properties": {
        "quadkey":       {"type": "keyword"},
        "network":       {"type": "keyword"},
        "period":        {"type": "date", "format": "yyyy-MM-dd"},
        "year_quarter":  {"type": "keyword"},
        "country_iso3":  {"type": "keyword"},
        "country":       {"type": "keyword"},
        "admin1":        {"type": "keyword"},
        "settlement":    {"type": "keyword"},
        "speed_class":   {"type": "keyword"},

        "tile_geom":     {"type": "geo_shape"},      # <- l'objet de ce TP
        "centroid":      {"type": "geo_point"},

        "download_mbps": {"type": "float"},
        "upload_mbps":   {"type": "float"},
        "latency_ms":    {"type": "float"},
        "tests":         {"type": "integer"},
        "devices":       {"type": "integer"},
        "population":    {"type": "float"},
        "pop_density":   {"type": "float"},
        "tile_km2":      {"type": "float"},
    },
}

if es.indices.exists(index=INDEX_TILES):
    es.indices.delete(index=INDEX_TILES)
es.indices.create(index=INDEX_TILES, settings=TILES_SETTINGS, mappings=TILES_MAPPINGS)
print(f"Index '{INDEX_TILES}' cree.")
display(HTML(f"<pre style='background:{AFDB['mist']};border:1px solid {AFDB['sage']};"
             f"border-radius:10px;padding:14px;font-size:12px;'>"
             + json.dumps(es.indices.get_mapping(index=INDEX_TILES)
                            .body[INDEX_TILES]["mappings"]["properties"]["tile_geom"], indent=2)
             + "</pre>"))

In [ ]:
# -----------------------------------------------------------------------------
#  Indexation en masse (_bulk)
# -----------------------------------------------------------------------------
def tile_actions(df):
    for r in df.to_dict("records"):
        yield {
            "_index": INDEX_TILES,
            "_id": f"{r['network']}_{r['period']}_{r['quadkey']}",
            "_source": {
                "quadkey": r["quadkey"], "network": r["network"],
                "period": r["period"], "year_quarter": r["year_quarter"],
                "country_iso3": ISO, "country": COUNTRY_NAME,
                "admin1": r["admin1"], "settlement": r["settlement"],
                "speed_class": r["speed_class"],
                # Envelope = [[coin haut-gauche], [coin bas-droit]] en [lon, lat]
                "tile_geom": {"type": "envelope",
                              "coordinates": [[round(r["min_lon"], 6), round(r["max_lat"], 6)],
                                              [round(r["max_lon"], 6), round(r["min_lat"], 6)]]},
                "centroid": {"lat": round(r["lat"], 6), "lon": round(r["lon"], 6)},
                "download_mbps": round(float(r["download_mbps"]), 3),
                "upload_mbps":   round(float(r["upload_mbps"]), 3),
                "latency_ms":    round(float(r["latency_ms"]), 2),
                "tests": int(r["tests"]), "devices": int(r["devices"]),
                "population":  round(float(r["population"]), 2),
                "pop_density": round(float(r["pop_density"]), 2),
                "tile_km2":    round(float(r["tile_km2"]), 5),
            },
        }

t0, ok_count, errors = time.time(), 0, []
with tqdm(total=len(tiles), desc="Indexation _bulk", unit="doc") as bar:
    for ok, item in helpers.streaming_bulk(es, tile_actions(tiles), chunk_size=BULK_CHUNK,
                                           max_retries=3, raise_on_error=False,
                                           request_timeout=180):
        ok_count += int(ok)
        if not ok: errors.append(item)
        bar.update(1)

es.indices.refresh(index=INDEX_TILES)
elapsed = max(time.time() - t0, 1e-9)
size = es.indices.stats(index=INDEX_TILES)["indices"][INDEX_TILES]["total"]["store"]["size_in_bytes"]

kpi_row([
    (f"{es.count(index=INDEX_TILES)['count']:,}", "Documents indexes",   AFDB["green"]),
    (f"{ok_count/elapsed:,.0f}",                  "Documents / seconde", AFDB["deep"]),
    (f"{size/1e6:,.1f} Mo",                       "Taille de l'index",   AFDB["teal"]),
    (f"{len(errors):,}", "Erreurs", AFDB["brick"] if errors else AFDB["slate"]),
])
if errors:
    print("Exemple d'erreur :", json.dumps(errors[0], indent=2)[:600])

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">07 &middot; AGR&Eacute;GATIONS</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Agr&eacute;gations statistiques, temporelles et g&eacute;ospatiales</div>
</div>

Toutes les requ&ecirc;tes utilisent `size: 0`&nbsp;: on ne rapatrie **aucun document**, seulement les r&eacute;sultats calcul&eacute;s par le cluster. C'est le mode de travail normal &agrave; l'&eacute;chelle&nbsp;: la donn&eacute;e ne se d&eacute;place pas, le calcul va &agrave; elle.

Parcours&nbsp;: `stats` &rarr; `percentiles` &rarr; `weighted_avg` &rarr; `date_histogram` &rarr; `terms` imbriqu&eacute;es &rarr; `filter` &rarr; `geotile_grid` &rarr; `geo_bounding_box` &rarr; `geo_shape` &rarr; `geo_distance`.

In [ ]:
# -----------------------------------------------------------------------------
#  7.1  Statistiques descriptives et percentiles, par type de reseau
# -----------------------------------------------------------------------------
PRIMARY_NET = "fixed" if "fixed" in set(tiles["network"]) else sorted(set(tiles["network"]))[0]

resp = es.search(index=INDEX_TILES, size=0, aggs={
    "par_reseau": {
        "terms": {"field": "network"},
        "aggs": {
            "debit":       {"stats": {"field": "download_mbps"}},
            "percentiles": {"percentiles": {"field": "download_mbps",
                                            "percents": [10, 25, 50, 75, 90, 95]}},
            "latence":     {"percentiles": {"field": "latency_ms", "percents": [50, 90]}},
            "tests":       {"sum": {"field": "tests"}},
            # Moyenne ponderee par le nombre de tests : une tuile a 3 tests ne pese pas
            # autant qu'une tuile a 3 000 tests.
            "debit_pondere": {"weighted_avg": {"value": {"field": "download_mbps"},
                                               "weight": {"field": "tests"}}},
        }}})

rows = []
for b in resp["aggregations"]["par_reseau"]["buckets"]:
    p = b["percentiles"]["values"]
    rows.append({"reseau": b["key"], "tuiles": b["doc_count"], "tests": int(b["tests"]["value"]),
                 "moy_simple": b["debit"]["avg"], "moy_ponderee": b["debit_pondere"]["value"],
                 "p10": p["10.0"], "median": p["50.0"], "p90": p["90.0"], "p95": p["95.0"],
                 "latence_p50": b["latence"]["values"]["50.0"]})
display(pd.DataFrame(rows).set_index("reseau").style.format("{:,.1f}").set_caption(
    f"Debit descendant (Mbps) et latence (ms) — {COUNTRY_NAME}, {len(QUARTER_LABELS)} trimestres"))

card("Lecture",
     "L'&eacute;cart entre <b>moyenne simple</b> et <b>moyenne pond&eacute;r&eacute;e par les tests</b> mesure un "
     "biais&nbsp;: quand la pond&eacute;r&eacute;e est nettement sup&eacute;rieure, les zones les plus test&eacute;es "
     "(donc les plus urbaines) sont aussi les mieux servies.", tone="warn")

In [ ]:
# -----------------------------------------------------------------------------
#  7.2  date_histogram : l'agregation temporelle native d'Elasticsearch
# -----------------------------------------------------------------------------
resp = es.search(index=INDEX_TILES, size=0, aggs={
    "trimestres": {
        "date_histogram": {"field": "period", "calendar_interval": "quarter",
                           "min_doc_count": 1, "format": "yyyy-MM-dd"},
        "aggs": {
            "par_reseau": {
                "terms": {"field": "network"},
                "aggs": {"median": {"percentiles": {"field": "download_mbps", "percents": [50]}},
                         "tests":  {"sum": {"field": "tests"}},
                         "pop_10": {"filter": {"range": {"download_mbps": {"gte": 10}}},
                                    "aggs": {"pop": {"sum": {"field": "population"}}}},
                         "pop":    {"sum": {"field": "population"}}}}}}})

evo = []
for b in resp["aggregations"]["trimestres"]["buckets"]:
    for nb in b["par_reseau"]["buckets"]:
        pop = nb["pop"]["value"] or np.nan
        evo.append({"periode": b["key_as_string"], "reseau": nb["key"],
                    "tuiles": nb["doc_count"],
                    "median_dl": nb["median"]["values"]["50.0"],
                    "tests": nb["tests"]["value"],
                    "pct_pop_10mbps": 100 * nb["pop_10"]["pop"]["value"] / pop})
evo = pd.DataFrame(evo)
display(evo.pivot(index="periode", columns="reseau",
                  values="median_dl").style.format("{:,.1f}").set_caption(
                  "Debit descendant median (Mbps) par trimestre"))

card("Pourquoi c'est important",
     "<code>calendar_interval: quarter</code> laisse Elasticsearch g&eacute;rer les bornes de "
     "trimestre&nbsp;: pas de calcul de dates c&ocirc;t&eacute; client, pas d'erreur de fuseau. C'est "
     "aussi ce champ <code>period</code> qui rend le s&eacute;lecteur de dates de Kibana op&eacute;rant "
     "sur l'ensemble du tableau de bord.", tone="ok")

In [ ]:
# -----------------------------------------------------------------------------
#  Visualisation 1 : evolution trimestrielle
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13.6, 4.4))
palette = {"fixed": AFDB["green"], "mobile": AFDB["teal"]}

for net, grp in evo.groupby("reseau"):
    g = grp.sort_values("periode")
    axes[0].plot(g["periode"], g["median_dl"], marker="o", linewidth=2.6, markersize=7,
                 color=palette.get(net, AFDB["deep"]), label=net)
    axes[1].plot(g["periode"], g["pct_pop_10mbps"], marker="o", linewidth=2.6, markersize=7,
                 color=palette.get(net, AFDB["deep"]), label=net)

axes[0].set_ylabel("Mbps"); axes[0].legend()
chart(axes[0], "07 · SERIE TEMPORELLE", "Debit descendant median par trimestre",
      "Source : Ookla Open Data, agregation date_histogram cote Elasticsearch.")
axes[1].set_ylabel("% de la population mesuree"); axes[1].legend()
chart(axes[1], "07 · COUVERTURE", "Population mesuree au-dessus de 10 Mbps",
      "Sources : Ookla Open Data, WorldPop. Calcul de l'auteur.")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# -----------------------------------------------------------------------------
#  7.3  Agregation imbriquee : classement des regions (dernier trimestre)
# -----------------------------------------------------------------------------
resp = es.search(index=INDEX_TILES, size=0,
    query={"bool": {"filter": [{"term": {"network": PRIMARY_NET}},
                               {"term": {"year_quarter": LATEST_Q}}]}},
    aggs={"regions": {
        "terms": {"field": "admin1", "size": 60, "order": {"median_dl.50": "desc"}},
        "aggs": {
            "median_dl":  {"percentiles": {"field": "download_mbps", "percents": [50]}},
            "median_up":  {"percentiles": {"field": "upload_mbps",  "percents": [50]}},
            "median_lat": {"percentiles": {"field": "latency_ms",   "percents": [50]}},
            "population": {"sum": {"field": "population"}},
            "tests":      {"sum": {"field": "tests"}},
            "pop_10":     {"filter": {"range": {"download_mbps": {"gte": 10}}},
                           "aggs": {"pop": {"sum": {"field": "population"}}}},
            "pop_100":    {"filter": {"range": {"download_mbps": {"gte": 100}}},
                           "aggs": {"pop": {"sum": {"field": "population"}}}},
        }}})

reg = pd.DataFrame([{
    "admin1": b["key"], "tuiles": b["doc_count"], "tests": int(b["tests"]["value"]),
    "median_dl": b["median_dl"]["values"]["50.0"],
    "median_up": b["median_up"]["values"]["50.0"],
    "median_lat": b["median_lat"]["values"]["50.0"],
    "pop_mesuree": b["population"]["value"],
    "pop_10": b["pop_10"]["pop"]["value"], "pop_100": b["pop_100"]["pop"]["value"],
} for b in resp["aggregations"]["regions"]["buckets"]])
reg["pct_10mbps"]  = 100 * reg["pop_10"]  / reg["pop_mesuree"].replace(0, np.nan)
reg["pct_100mbps"] = 100 * reg["pop_100"] / reg["pop_mesuree"].replace(0, np.nan)
display(reg.head(12)[["admin1","tuiles","median_dl","median_up","median_lat",
                      "pct_10mbps","pct_100mbps"]])

In [ ]:
# -----------------------------------------------------------------------------
#  Visualisation 2 : classement des regions + effet de la densite
# -----------------------------------------------------------------------------
fig = plt.figure(figsize=(14, 5.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 1])
ax0, ax1 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

top = reg.dropna(subset=["median_dl"]).head(12).sort_values("median_dl")
colors = [RAMP[min(len(RAMP)-1, int(i / max(len(top)-1, 1) * (len(RAMP)-1)))]
          for i in range(len(top))]
bars = ax0.barh(top["admin1"], top["median_dl"], color=colors, edgecolor="none", height=0.68)
for b, v in zip(bars, top["median_dl"]):
    ax0.text(v + max(top["median_dl"]) * 0.012, b.get_y() + b.get_height()/2,
             f"{v:,.1f}", va="center", fontsize=9.5, fontweight="bold", color=AFDB["ink"])
ax0.set_xlabel("Debit descendant median (Mbps)")
ax0.xaxis.grid(True); ax0.yaxis.grid(False)
ax0.set_xlim(0, max(top["median_dl"]) * 1.16)
chart(ax0, f"07 · {PRIMARY_NET.upper()} · {LATEST_Q}", f"Debit median par region",
      "Source : Ookla Open Data, agrege par Elasticsearch.")

sub = tiles[(tiles["network"] == PRIMARY_NET) & (tiles["year_quarter"] == LATEST_Q) &
            (tiles["pop_density"] > 0)]
sub = sub.sample(min(8000, len(sub)), random_state=7)
ax1.scatter(sub["pop_density"], sub["download_mbps"].clip(upper=400), s=6,
            alpha=0.25, color=AFDB["deep"], edgecolors="none")
ax1.set_xscale("log")
ax1.set_xlabel("Densite de population (hab/km², log)"); ax1.set_ylabel("Debit (Mbps)")
chart(ax1, "07 · CORRELATION", "Densite de peuplement et debit",
      "Chaque point est une tuile de ~610 m. Sources : Ookla Open Data, WorldPop.")
plt.tight_layout(); plt.show()

### 7.4 &mdash; Agr&eacute;gations et requ&ecirc;tes **g&eacute;ospatiales**

C'est ici que le mapping paie. Quatre m&eacute;canismes compl&eacute;mentaires&nbsp;:

* **`geotile_grid`** &mdash; agr&egrave;ge sur la grille Web Mercator elle-m&ecirc;me. Les cl&eacute;s sont des `z/x/y`&nbsp;: la logique du quadkey, c&ocirc;t&eacute; serveur. C'est ce qui alimente les cartes de chaleur de Kibana Maps.
* **`geo_bounding_box`** &mdash; filtre rectangulaire sur `geo_point`, tr&egrave;s rapide (comparaison de bornes dans le BKD-tree).
* **`geo_shape` + `relation`** &mdash; filtre sur une g&eacute;om&eacute;trie quelconque&nbsp;: `intersects` (d&eacute;faut), `within`, `contains`, `disjoint`.
* **`geo_distance`** &mdash; filtre circulaire autour d'un point, indispensable pour les analyses de rayon.

In [ ]:
# -----------------------------------------------------------------------------
#  7.4.a  geotile_grid
# -----------------------------------------------------------------------------
GRID_Z = 8
resp = es.search(index=INDEX_TILES, size=0,
    query={"bool": {"filter": [{"term": {"network": PRIMARY_NET}},
                               {"term": {"year_quarter": LATEST_Q}}]}},
    aggs={"grille": {"geotile_grid": {"field": "centroid", "precision": GRID_Z, "size": 5000},
                     "aggs": {"debit": {"avg": {"field": "download_mbps"}},
                              "pop":   {"sum": {"field": "population"}}}}})

g = []
for b in resp["aggregations"]["grille"]["buckets"]:
    z, x, y = map(int, b["key"].split("/"))
    n = 2 ** z
    g.append({"lon": (x + 0.5) / n * 360.0 - 180.0,
              "lat": math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + 0.5) / n)))),
              "tuiles": b["doc_count"], "debit": b["debit"]["value"], "pop": b["pop"]["value"]})
grid = pd.DataFrame(g)
print(f"{len(grid)} cellules z{GRID_Z} (≈ {40075/2**GRID_Z:,.0f} km a l'equateur)")

fig, ax = plt.subplots(figsize=(7.4, 7.0))
adm1.boundary.plot(ax=ax, color=AFDB["sage"], linewidth=0.7)
sc = ax.scatter(grid["lon"], grid["lat"],
                s=np.clip(grid["tuiles"] / grid["tuiles"].max() * 420, 14, 420),
                c=grid["debit"], cmap=mpl.colors.LinearSegmentedColormap.from_list("afdb", RAMP),
                alpha=0.88, edgecolors="white", linewidths=0.4)
cb = plt.colorbar(sc, ax=ax, shrink=0.72, pad=0.02)
cb.set_label("Debit moyen (Mbps)", color=AFDB["slate"], fontsize=9.5)
cb.outline.set_edgecolor(AFDB["sage"])
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.grid(alpha=0.35)
ax.set_aspect(1 / math.cos(math.radians(float(np.mean(grid["lat"])))))
chart(ax, f"07 · GEOTILE_GRID z{GRID_Z}", f"Agregation geospatiale cote cluster — {LATEST_Q}",
      "Taille du cercle = nombre de tuiles. Source : Ookla Open Data.")
plt.tight_layout(); plt.show()

In [ ]:
# -----------------------------------------------------------------------------
#  7.4.b  geo_bounding_box, geo_shape et geo_distance
# -----------------------------------------------------------------------------
last = tiles[(tiles["year_quarter"] == LATEST_Q) & (tiles["network"] == PRIMARY_NET)]
hub = last.loc[last["pop_density"].idxmax()] if HAS_POP else last.loc[last["tests"].idxmax()]
HUB_LAT, HUB_LON = float(hub["lat"]), float(hub["lon"])
print(f"Centre urbain principal detecte : {hub['admin1']}  ({HUB_LAT:.4f}, {HUB_LON:.4f})")

D = 0.25
q_bbox = {"geo_bounding_box": {"centroid": {
    "top_left":     {"lat": HUB_LAT + D, "lon": HUB_LON - D},
    "bottom_right": {"lat": HUB_LAT - D, "lon": HUB_LON + D}}}}

top_region = reg.iloc[0]["admin1"] if len(reg) else adm1.iloc[0]["admin1"]
geom_region = adm1.loc[adm1["admin1"] == top_region, "geometry"].iloc[0].simplify(0.01).buffer(0)
q_shape = {"geo_shape": {"tile_geom": {
    "shape": json.loads(gpd.GeoSeries([geom_region]).to_json())["features"][0]["geometry"],
    "relation": "intersects"}}}

q_dist = {"geo_distance": {"distance": "25km", "centroid": {"lat": HUB_LAT, "lon": HUB_LON}}}

AGGS = {"debit": {"percentiles": {"field": "download_mbps", "percents": [50]}},
        "pop":   {"sum": {"field": "population"}},
        "tests": {"sum": {"field": "tests"}}}

results = []
for label, q in [("Pays entier", {"match_all": {}}),
                 ("geo_bounding_box (±25 km du hub)", q_bbox),
                 (f"geo_shape intersects ({str(top_region)[:20]})", q_shape),
                 ("geo_distance 25 km du hub", q_dist)]:
    t0 = time.time()
    r = es.search(index=INDEX_TILES, size=0, aggs=AGGS,
                  query={"bool": {"filter": [q, {"term": {"network": PRIMARY_NET}},
                                             {"term": {"year_quarter": LATEST_Q}}]}})
    results.append({"requete": label, "tuiles": r["hits"]["total"]["value"],
                    "debit_median": r["aggregations"]["debit"]["values"]["50.0"],
                    "population": r["aggregations"]["pop"]["value"],
                    "tests": r["aggregations"]["tests"]["value"],
                    "ms": (time.time() - t0) * 1000})
display(pd.DataFrame(results).set_index("requete").style.format(
    {"tuiles": "{:,.0f}", "debit_median": "{:,.1f}", "population": "{:,.0f}",
     "tests": "{:,.0f}", "ms": "{:,.0f} ms"}))

card("Ce qu'il faut retenir",
     "<code>geo_bounding_box</code> et <code>geo_distance</code> travaillent sur le "
     "<code>geo_point</code>&nbsp;: rapides, mais ils testent le <b>centre</b> de la tuile. "
     "<code>geo_shape intersects</code> travaille sur la g&eacute;om&eacute;trie r&eacute;elle&nbsp;: une tuile "
     "&agrave; cheval sur la fronti&egrave;re est compt&eacute;e. Le bon choix d&eacute;pend de la question "
     "pos&eacute;e, pas de la performance.", tone="ok")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">08 &middot; ANALYSE</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Indice de fracture num&eacute;rique et index de r&eacute;sultats</div>
</div>

On produit un **second index**, `ookla-admin-…`, &agrave; la granularit&eacute; *r&eacute;gion &times; r&eacute;seau &times; trimestre*. Peu de documents, mais riches, avec la g&eacute;om&eacute;trie ADM1 en `geo_shape` **polygone** &mdash; ce qui permet la carte choropl&egrave;the dans Kibana et le suivi dans le temps.

**Indice de fracture num&eacute;rique (0 = aucun retard, 100 = retard maximal)**&nbsp;:

$$\text{IFN} = 100 \times \Big[\;0{,}50\,(1 - p_{\ge 10}) \;+\; 0{,}30\,\big(1 - \min(\tfrac{\tilde{d}}{50}, 1)\big) \;+\; 0{,}20\,(1 - c)\;\Big]$$

o&ugrave; $p_{\ge 10}$ est la part de population mesur&eacute;e au-dessus de 10&nbsp;Mbps, $\tilde{d}$ le d&eacute;bit m&eacute;dian et $c$ le taux de couverture.

> **Avertissement m&eacute;thodologique.** Les tuiles Ookla ne couvrent que les zones o&ugrave; des tests ont &eacute;t&eacute; r&eacute;alis&eacute;s&nbsp;: l'absence de tuile signifie &laquo;&nbsp;pas de mesure&nbsp;&raquo;, pas &laquo;&nbsp;pas de r&eacute;seau&nbsp;&raquo;. Le taux de couverture se lit comme un indicateur d'**observabilit&eacute;**. Les pond&eacute;rations sont un choix d'atelier, &agrave; calibrer avec les &eacute;quipes sectorielles avant tout usage d&eacute;cisionnel.

In [ ]:
# -----------------------------------------------------------------------------
#  8.1  Documents region x reseau x trimestre
# -----------------------------------------------------------------------------
from shapely.geometry import mapping as shp_mapping
from shapely.geometry.polygon import orient

GEOM_CACHE = {}
for _, row in adm1.iterrows():
    gm = row.geometry.simplify(0.005).buffer(0)
    GEOM_CACHE[row["admin1"]] = (
        shp_mapping(orient(gm, sign=1.0) if gm.geom_type == "Polygon" else gm),
        {"lat": round(float(gm.centroid.y), 6), "lon": round(float(gm.centroid.x), 6)})

def build_admin_docs():
    docs = []
    for (net, yq), grp_all in tiles.groupby(["network", "year_quarter"]):
        period = grp_all["period"].iloc[0]
        for adm, grp in grp_all.groupby("admin1"):
            pop_tot  = ADMIN_POP.get(adm, float("nan"))
            pop_meas = float(grp["population"].sum())
            p10  = float(grp.loc[grp["download_mbps"] >= 10,  "population"].sum())
            p25  = float(grp.loc[grp["download_mbps"] >= 25,  "population"].sum())
            p100 = float(grp.loc[grp["download_mbps"] >= 100, "population"].sum())
            med  = float(grp["download_mbps"].median())
            cov   = (pop_meas / pop_tot) if (pop_tot == pop_tot and pop_tot > 0) else np.nan
            pct10 = (p10 / pop_meas) if pop_meas > 0 else np.nan
            ifn = 100 * (0.50 * (1 - (pct10 if pct10 == pct10 else 0))
                       + 0.30 * (1 - min(med / 50.0, 1.0))
                       + 0.20 * (1 - (min(cov, 1.0) if cov == cov else 0)))
            geom, centroid = GEOM_CACHE[adm]
            docs.append({
                "admin1": adm, "network": net, "country": COUNTRY_NAME, "country_iso3": ISO,
                "period": period, "year_quarter": yq,
                "population": None if pop_tot != pop_tot else round(pop_tot, 0),
                "pop_measured": round(pop_meas, 0),
                "coverage_rate": None if cov != cov else round(float(cov) * 100, 2),
                "pop_above_10mbps": round(p10, 0),
                "pct_pop_10mbps":  None if pct10 != pct10 else round(pct10 * 100, 2),
                "pct_pop_25mbps":  round(100 * p25 / pop_meas, 2) if pop_meas > 0 else None,
                "pct_pop_100mbps": round(100 * p100 / pop_meas, 2) if pop_meas > 0 else None,
                "median_download_mbps": round(med, 2),
                "median_upload_mbps":   round(float(grp["upload_mbps"].median()), 2),
                "median_latency_ms":    round(float(grp["latency_ms"].median()), 2),
                "weighted_download_mbps": round(float(
                    np.average(grp["download_mbps"], weights=grp["tests"].clip(lower=1))), 2),
                "tiles": int(len(grp)), "tests": int(grp["tests"].sum()),
                "devices": int(grp["devices"].sum()),
                "digital_divide_index": round(float(ifn), 1),
                "centroid": centroid, "geometry": geom,
            })
    return docs

ADMIN_DOCS = build_admin_docs()
admin_df = pd.DataFrame([{k: v for k, v in d.items() if k not in ("geometry", "centroid")}
                         for d in ADMIN_DOCS])
print(f"{len(ADMIN_DOCS)} documents : {admin_df['admin1'].nunique()} regions x "
      f"{admin_df['network'].nunique()} reseaux x {admin_df['year_quarter'].nunique()} trimestres")
display(admin_df[(admin_df["network"] == PRIMARY_NET) & (admin_df["year_quarter"] == LATEST_Q)]
        .sort_values("digital_divide_index", ascending=False)
        .head(10)[["admin1","population","coverage_rate","median_download_mbps",
                   "pct_pop_10mbps","digital_divide_index"]])

In [ ]:
# -----------------------------------------------------------------------------
#  8.2  Indexation des resultats d'analyse
# -----------------------------------------------------------------------------
ADMIN_MAPPINGS = {
    "dynamic": "strict",
    "properties": {
        "admin1": {"type": "keyword"}, "network": {"type": "keyword"},
        "country": {"type": "keyword"}, "country_iso3": {"type": "keyword"},
        "period": {"type": "date", "format": "yyyy-MM-dd"},
        "year_quarter": {"type": "keyword"},
        "geometry": {"type": "geo_shape"}, "centroid": {"type": "geo_point"},
        "population": {"type": "float"}, "pop_measured": {"type": "float"},
        "coverage_rate": {"type": "float"}, "pop_above_10mbps": {"type": "float"},
        "pct_pop_10mbps": {"type": "float"}, "pct_pop_25mbps": {"type": "float"},
        "pct_pop_100mbps": {"type": "float"},
        "median_download_mbps": {"type": "float"}, "median_upload_mbps": {"type": "float"},
        "median_latency_ms": {"type": "float"}, "weighted_download_mbps": {"type": "float"},
        "tiles": {"type": "integer"}, "tests": {"type": "long"}, "devices": {"type": "long"},
        "digital_divide_index": {"type": "float"},
    },
}

if es.indices.exists(index=INDEX_ADMIN):
    es.indices.delete(index=INDEX_ADMIN)
es.indices.create(index=INDEX_ADMIN,
                  settings={"number_of_shards": 1, "number_of_replicas": 0},
                  mappings=ADMIN_MAPPINGS)

ok, errs = helpers.bulk(
    es, ({"_index": INDEX_ADMIN,
          "_id": f"{d['network']}_{d['period']}_{d['admin1']}", "_source": d}
         for d in ADMIN_DOCS),
    raise_on_error=False, request_timeout=180)
es.indices.refresh(index=INDEX_ADMIN)

kpi_row([
    (f"{es.count(index=INDEX_TILES)['count']:,}", f"Documents · tuiles",   AFDB["green"]),
    (f"{es.count(index=INDEX_ADMIN)['count']:,}", f"Documents · analyse",  AFDB["deep"]),
    (f"{len(QUARTER_LABELS)}",                    "Trimestres couverts",   AFDB["teal"]),
    (f"{len(errs)}", "Erreurs", AFDB["brick"] if errs else AFDB["slate"]),
])
if errs: print(json.dumps(errs[0], indent=2)[:600])

In [ ]:
# -----------------------------------------------------------------------------
#  Visualisation 3 : cartes choroplethes (dernier trimestre)
# -----------------------------------------------------------------------------
sel = admin_df[(admin_df["network"] == PRIMARY_NET) & (admin_df["year_quarter"] == LATEST_Q)]
m = adm1.merge(sel[["admin1", "digital_divide_index", "median_download_mbps"]],
               on="admin1", how="left")

fig, axes = plt.subplots(1, 2, figsize=(14, 6.4))
cmap_g = mpl.colors.LinearSegmentedColormap.from_list("afdb_g", RAMP)
cmap_r = mpl.colors.LinearSegmentedColormap.from_list(
    "afdb_r", [AFDB["mint"], AFDB["gold"], AFDB["terra"], AFDB["brick"]])

for ax, col, cmap, lbl, kick, ttl in [
    (axes[0], "median_download_mbps", cmap_g, "Debit median (Mbps)", "08 · PERFORMANCE",
     "Debit descendant median par region"),
    (axes[1], "digital_divide_index", cmap_r, "Indice (0–100)", "08 · EQUITE",
     "Indice de fracture numerique"),
]:
    m.plot(column=col, cmap=cmap, ax=ax, edgecolor="white", linewidth=0.7, legend=True,
           legend_kwds={"shrink": 0.68, "label": lbl},
           missing_kwds={"color": AFDB["sage"], "label": "Sans donnee"})
    ax.set_axis_off()
    ax.text(0, 1.06, kick, transform=ax.transAxes, fontsize=9, fontweight="bold", color=AFDB["deep"])
    ax.text(0, 1.005, ttl, transform=ax.transAxes, fontsize=13.5, fontweight="bold", color=AFDB["ink"])

fig.text(0.01, 0.015, f"Sources : Ookla Open Data {LATEST_Q} (reseau {PRIMARY_NET}), "
         f"WorldPop {WORLDPOP_YEAR}, geoBoundaries ADM1. Indice composite : calcul de l'auteur.",
         fontsize=8.5, style="italic", color=AFDB["slate"])
plt.tight_layout(rect=[0, 0.035, 1, 1]); plt.show()

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">09 &middot; KIBANA</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">G&eacute;n&eacute;rer un tableau de bord de 13 panneaux</div>
</div>

Un dashboard Kibana est un **saved object** JSON. On peut donc le produire par code, le versionner dans Git, et le rejouer &agrave; l'identique sur un autre pays ou un autre cluster.

**Deux d&eacute;cisions de conception m&eacute;ritent une explication.**

*Pourquoi des visualisations &laquo;&nbsp;agg-based&nbsp;&raquo; et non Lens.* Le sch&eacute;ma JSON interne de Lens &eacute;volue d'une version mineure de Kibana &agrave; l'autre&nbsp;; un objet g&eacute;n&eacute;r&eacute; pour 8.15 peut &ecirc;tre refus&eacute; ailleurs. Le format `visualization` + `visState` est stable depuis Kibana&nbsp;6 et reste pris en charge. Le tableau de bord est donc reproductible sur un cluster dont vous ne ma&icirc;trisez pas la version &mdash; et chaque panneau offre un bouton *Convert to Lens* si vous voulez ensuite l'&eacute;diter dans l'&eacute;diteur moderne.

*Pourquoi les data views d&eacute;clarent maintenant un champ temporel.* Avec un seul trimestre, il fallait d&eacute;sactiver le filtre de dates pour que les panneaux ne soient pas vides. Avec une s&eacute;rie, `period` devient le **champ temporel** des deux data views&nbsp;: le s&eacute;lecteur en haut du dashboard filtre alors l'ensemble des panneaux, et le dashboard m&eacute;morise sa propre plage par d&eacute;faut (`timeRestore`) pour s'ouvrir sur des donn&eacute;es.

In [ ]:
# -----------------------------------------------------------------------------
#  9.1  Data views, avec champ temporel
# -----------------------------------------------------------------------------
KB_H = {"kbn-xsrf": "true", "Content-Type": "application/json"}
DV_TILES, DV_ADMIN = f"dv-{INDEX_TILES}", f"dv-{INDEX_ADMIN}"

def create_data_view(dv_id, index, name, time_field="period"):
    r = requests.post(f"{KIBANA_URL}/api/data_views/data_view", headers=KB_H,
                      json={"override": True,
                            "data_view": {"id": dv_id, "title": index, "name": name,
                                          "timeFieldName": time_field}},
                      timeout=120)
    ok = r.status_code in (200, 201)
    print(f"  data view {name:<30} {'OK' if ok else 'ECHEC ' + r.text[:200]}")
    return ok

assert service_up(KIBANA_URL, "/api/status"), "Kibana n'est pas joignable."
create_data_view(DV_TILES, INDEX_TILES, f"Ookla · tuiles · {ISO}")
create_data_view(DV_ADMIN, INDEX_ADMIN, f"Ookla · regions · {ISO}")

In [ ]:
# -----------------------------------------------------------------------------
#  9.2  Fabriques de visualisations agg-based
#
#  Un objet = attributes.visState (la definition) + kibanaSavedObjectMeta
#  (la source de donnees, par reference vers la data view) + uiStateJSON
#  (les couleurs par serie : c'est la que la charte AfDB s'applique).
# -----------------------------------------------------------------------------
def vis_object(so_id, title, vis_state, dv_id, kuery="", colors=None):
    return {
        "id": so_id, "type": "visualization",
        "attributes": {
            "title": title, "description": "",
            "visState": json.dumps(vis_state, ensure_ascii=False),
            "uiStateJSON": json.dumps({"vis": {"colors": colors}} if colors else {}),
            "kibanaSavedObjectMeta": {"searchSourceJSON": json.dumps({
                "query": {"language": "kuery", "query": kuery}, "filter": [],
                "indexRefName": "kibanaSavedObjectMeta.searchSourceJSON.index"})},
        },
        "references": [{"name": "kibanaSavedObjectMeta.searchSourceJSON.index",
                        "type": "index-pattern", "id": dv_id}],
    }

def agg_metric(idx, agg_type, field, label):
    params = {"customLabel": label}
    if agg_type != "count":
        params["field"] = field
    if agg_type == "median":
        params["percents"] = [50]
    return {"id": str(idx), "enabled": True, "type": agg_type,
            "schema": "metric", "params": params}

def agg_terms(idx, field, size, order_by="1", order="desc", label=None):
    return {"id": str(idx), "enabled": True, "type": "terms", "schema": "segment",
            "params": {"field": field, "orderBy": order_by, "order": order, "size": size,
                       "otherBucket": False, "otherBucketLabel": "Autres",
                       "missingBucket": False, "missingBucketLabel": "N/A",
                       "customLabel": label or ""}}

def agg_group(idx, field, size=5):
    d = agg_terms(idx, field, size, order_by="_key", order="asc")
    d["schema"] = "group"
    return d

def vs_metric(label, agg_type, field, font=52):
    return {"title": label, "type": "metric",
            "aggs": [agg_metric(1, agg_type, field, label)],
            "params": {"addTooltip": True, "addLegend": False, "type": "metric",
                       "metric": {"percentageMode": False, "useRanges": False,
                                  "colorSchema": "Green to Red", "metricColorMode": "None",
                                  "colorsRange": [{"from": 0, "to": 10000}],
                                  "labels": {"show": True}, "invertColors": False,
                                  "style": {"bgFill": "#000", "bgColor": False,
                                            "labelColor": False, "subText": "",
                                            "fontSize": font}}}}

def _axes(kind, label, horizontal):
    return {
        "type": kind, "grid": {"categoryLines": False},
        "categoryAxes": [{"id": "CategoryAxis-1", "type": "category",
                          "position": "left" if horizontal else "bottom", "show": True,
                          "style": {}, "scale": {"type": "linear"},
                          "labels": {"show": True, "filter": False, "truncate": 120,
                                     "rotate": 0}, "title": {}}],
        "valueAxes": [{"id": "ValueAxis-1", "name": "LeftAxis-1", "type": "value",
                       "position": "bottom" if horizontal else "left", "show": True,
                       "style": {}, "scale": {"type": "linear", "mode": "normal"},
                       "labels": {"show": True, "rotate": 0, "filter": True, "truncate": 100},
                       "title": {"text": label}}],
        "addTooltip": True, "addLegend": False, "legendPosition": "right",
        "times": [], "addTimeMarker": False,
        "labels": {"show": True}, "thresholdLine": {"show": False, "value": 10, "width": 1,
                                                    "style": "full", "color": "#E7664C"},
        "palette": {"type": "palette", "name": "default"},
    }

def vs_bar(label, metric_agg, metric_field, bucket_field, size, horizontal=True,
           order="desc", order_by="1"):
    kind = "horizontal_bar" if horizontal else "histogram"
    p = _axes(kind, label, horizontal)
    p["seriesParams"] = [{"show": True, "type": "histogram", "mode": "normal",
                          "data": {"label": label, "id": "1"}, "valueAxis": "ValueAxis-1",
                          "drawLinesBetweenPoints": True, "lineWidth": 2, "showCircles": True}]
    return {"title": label, "type": kind,
            "aggs": [agg_metric(1, metric_agg, metric_field, label),
                     agg_terms(2, bucket_field, size, order_by=order_by, order=order)],
            "params": p}

def vs_line(label, metric_agg, metric_field, x_field, split_field, size=24):
    p = _axes("line", label, horizontal=False)
    p["addLegend"] = True
    p["seriesParams"] = [{"show": True, "type": "line", "mode": "normal",
                          "data": {"label": label, "id": "1"}, "valueAxis": "ValueAxis-1",
                          "drawLinesBetweenPoints": True, "lineWidth": 3,
                          "interpolate": "linear", "showCircles": True}]
    return {"title": label, "type": "line",
            "aggs": [agg_metric(1, metric_agg, metric_field, label),
                     agg_terms(2, x_field, size, order_by="_key", order="asc"),
                     agg_group(3, split_field)],
            "params": p}

def vs_pie(label, metric_agg, metric_field, bucket_field, size=6):
    return {"title": label, "type": "pie",
            "aggs": [agg_metric(1, metric_agg, metric_field, label),
                     agg_terms(2, bucket_field, size, order_by="_key", order="asc")],
            "params": {"type": "pie", "addTooltip": True, "addLegend": True,
                       "legendPosition": "right", "isDonut": True, "distinctColors": True,
                       "labels": {"show": True, "values": True, "last_level": True,
                                  "truncate": 100, "position": "default", "valuesFormat": "percent"},
                       "palette": {"type": "palette", "name": "default"}}}

def vs_table(bucket_field, size, metrics):
    """metrics = [(agg_type, field, label), ...]"""
    aggs = [agg_terms(1, bucket_field, size, order_by="2", order="desc", label="Region")]
    for i, (a, f, l) in enumerate(metrics, start=2):
        aggs.append(agg_metric(i, a, f, l))
    return {"title": "Synthese", "type": "table", "aggs": aggs,
            "params": {"perPage": 12, "showPartialRows": False, "showTotal": False,
                       "showMetricsAtAllLevels": False, "totalFunc": "sum",
                       "percentageCol": "", "showToolbar": True,
                       "palette": {"type": "palette", "name": "default"}}}

print("Fabriques de visualisations pretes.")

In [ ]:
# -----------------------------------------------------------------------------
#  9.3  Les douze panneaux analytiques
# -----------------------------------------------------------------------------
P = f"ookla-{ISO.lower()}"
NET_Q = f'network : "{PRIMARY_NET}"'
objects = []

# --- Quatre indicateurs de tete ----------------------------------------------
objects += [
    vis_object(f"{P}-kpi-tests", "Tests de debit",
               vs_metric("Tests de debit", "sum", "tests"), DV_TILES),
    vis_object(f"{P}-kpi-dl", f"Debit median ({PRIMARY_NET})",
               vs_metric(f"Debit median {PRIMARY_NET} (Mbps)", "median", "download_mbps"),
               DV_TILES, kuery=NET_Q),
    vis_object(f"{P}-kpi-pop", "Population mesuree",
               vs_metric("Population sous tuile mesuree", "sum", "population"),
               DV_TILES, kuery=NET_Q),
    vis_object(f"{P}-kpi-lat", "Latence mediane",
               vs_metric("Latence mediane (ms)", "median", "latency_ms"),
               DV_TILES, kuery=NET_Q),
]

# --- Serie temporelle ---------------------------------------------------------
objects.append(vis_object(
    f"{P}-line-evo", "Evolution du debit median par trimestre",
    vs_line("Debit median (Mbps)", "median", "download_mbps", "year_quarter", "network"),
    DV_TILES, colors={"fixed": AFDB["green"], "mobile": AFDB["teal"]}))

# --- Repartition de la population par classe de debit -------------------------
objects.append(vis_object(
    f"{P}-pie-speed", "Population par classe de debit",
    vs_pie("Habitants", "sum", "population", "speed_class"),
    DV_TILES, kuery=NET_Q,
    colors={SPEED_CLASSES[0][2]: AFDB["brick"], SPEED_CLASSES[1][2]: AFDB["ochre"],
            SPEED_CLASSES[2][2]: RAMP[3],       SPEED_CLASSES[3][2]: AFDB["deep"]}))

# --- Classements regionaux ----------------------------------------------------
objects += [
    vis_object(f"{P}-bar-dl", "Debit median par region",
               vs_bar("Debit median (Mbps)", "median", "download_mbps", "admin1", 12),
               DV_TILES, kuery=NET_Q, colors={"Debit median (Mbps)": AFDB["green"]}),
    vis_object(f"{P}-bar-ifn", "Indice de fracture numerique",
               vs_bar("Indice de fracture", "max", "digital_divide_index", "admin1", 12),
               DV_ADMIN, kuery=NET_Q, colors={"Indice de fracture": AFDB["terra"]}),
    vis_object(f"{P}-bar-settlement", "Debit median par type de peuplement",
               vs_bar("Debit median (Mbps)", "median", "download_mbps", "settlement", 5,
                      horizontal=False, order="asc", order_by="_key"),
               DV_TILES, kuery=NET_Q, colors={"Debit median (Mbps)": AFDB["deep"]}),
    vis_object(f"{P}-bar-lat", "Latence mediane par region",
               vs_bar("Latence mediane (ms)", "median", "latency_ms", "admin1", 12),
               DV_TILES, kuery=NET_Q, colors={"Latence mediane (ms)": AFDB["ochre"]}),
]

# --- Tableau de synthese ------------------------------------------------------
objects.append(vis_object(
    f"{P}-table", "Synthese par region",
    vs_table("admin1", 40, [("max", "population", "Population"),
                            ("max", "median_download_mbps", "Debit median (Mbps)"),
                            ("max", "pct_pop_10mbps", "% pop. ≥ 10 Mbps"),
                            ("max", "median_latency_ms", "Latence (ms)"),
                            ("max", "digital_divide_index", "Indice de fracture")]),
    DV_ADMIN, kuery=NET_Q))

print(f"{len(objects)} visualisations generees.")

In [ ]:
# -----------------------------------------------------------------------------
#  9.4  Carte Maps : tuiles (MVT) + contours regionaux
# -----------------------------------------------------------------------------
center_lon, center_lat = (BBOX[0] + BBOX[2]) / 2, (BBOX[1] + BBOX[3]) / 2
span = max(BBOX[2] - BBOX[0], BBOX[3] - BBOX[1])
map_zoom = int(max(4, min(9, math.floor(math.log2(360.0 / max(span, 0.5))) + 1)))

basemap_layer = {
    "id": "lyr_base", "label": None, "minZoom": 0, "maxZoom": 24, "alpha": 1,
    "visible": True, "type": "EMS_VECTOR_TILE", "includeInFitToBounds": True,
    "style": {"type": "TILE"},
    "sourceDescriptor": {"type": "EMS_TMS", "isAutoSelect": True,
                         "lightModeDefault": "road_map_desaturated"},
}

tiles_layer = {
    "id": "lyr_tiles", "label": f"Tuiles Ookla — debit descendant ({PRIMARY_NET})",
    "minZoom": 0, "maxZoom": 24, "alpha": 0.85, "visible": True,
    "type": "MVT_VECTOR", "joins": [], "includeInFitToBounds": True,
    "query": {"language": "kuery", "query": NET_Q},
    "sourceDescriptor": {
        "type": "ES_SEARCH", "id": "src_tiles",
        "indexPatternRefName": "layer_2_source_index_pattern",
        "geoField": "tile_geom", "scalingType": "MVT", "filterByMapBounds": True,
        "tooltipProperties": ["quadkey", "admin1", "year_quarter", "download_mbps",
                              "upload_mbps", "latency_ms", "tests", "population"],
        "sortField": "", "sortOrder": "desc", "applyGlobalQuery": True,
        "applyGlobalTime": True, "applyForceRefresh": True,
    },
    "style": {"type": "VECTOR", "isTimeAware": True, "properties": {
        "fillColor": {"type": "DYNAMIC", "options": {
            "color": "Greens", "colorCategory": "palette_0", "type": "ORDINAL",
            "field": {"name": "download_mbps", "origin": "source"},
            "fieldMetaOptions": {"isEnabled": True, "sigma": 3}}},
        "lineColor": {"type": "STATIC", "options": {"color": "#FFFFFF"}},
        "lineWidth": {"type": "STATIC", "options": {"size": 0}},
        "iconSize": {"type": "STATIC", "options": {"size": 6}},
        "icon": {"type": "STATIC", "options": {"value": "marker"}},
        "symbolizeAs": {"options": {"value": "circle"}},
        "iconOrientation": {"type": "STATIC", "options": {"orientation": 0}},
        "labelText": {"type": "STATIC", "options": {"value": ""}},
        "labelColor": {"type": "STATIC", "options": {"color": "#000000"}},
        "labelSize": {"type": "STATIC", "options": {"size": 14}},
        "labelBorderColor": {"type": "STATIC", "options": {"color": "#FFFFFF"}},
        "labelBorderSize": {"options": {"size": "SMALL"}},
    }},
}

admin_layer = {
    "id": "lyr_adm", "label": "Regions (ADM1)", "minZoom": 0, "maxZoom": 24,
    "alpha": 1, "visible": True, "type": "GEOJSON_VECTOR", "joins": [],
    "includeInFitToBounds": True,
    "query": {"language": "kuery", "query": NET_Q},
    "sourceDescriptor": {
        "type": "ES_SEARCH", "id": "src_adm",
        "indexPatternRefName": "layer_3_source_index_pattern",
        "geoField": "geometry", "scalingType": "LIMIT", "topHitsSize": 1,
        "filterByMapBounds": False, "applyGlobalQuery": True, "applyGlobalTime": True,
        "sortField": "", "sortOrder": "desc",
        "tooltipProperties": ["admin1", "year_quarter", "population",
                              "median_download_mbps", "pct_pop_10mbps",
                              "digital_divide_index"],
    },
    "style": {"type": "VECTOR", "properties": {
        "fillColor": {"type": "STATIC", "options": {"color": "rgba(0,0,0,0)"}},
        "lineColor": {"type": "STATIC", "options": {"color": AFDB["deep"]}},
        "lineWidth": {"type": "STATIC", "options": {"size": 1.6}},
        "iconSize": {"type": "STATIC", "options": {"size": 6}},
        "symbolizeAs": {"options": {"value": "circle"}},
        "labelText": {"type": "STATIC", "options": {"value": ""}},
    }},
}

objects.append({
    "id": f"{P}-map", "type": "map",
    "attributes": {
        "title": f"Carte de connectivite — {COUNTRY_NAME}",
        "description": "Tuiles Ookla en geo_shape (MVT) et contours ADM1.",
        "layerListJSON": json.dumps([basemap_layer, admin_layer, tiles_layer]),
        "mapStateJSON": json.dumps({
            "zoom": map_zoom, "center": {"lon": center_lon, "lat": center_lat},
            "timeFilters": {"from": "now-5y", "to": "now"},
            "refreshConfig": {"isPaused": True, "interval": 0},
            "query": {"language": "kuery", "query": ""}, "filters": [],
            "settings": {"autoFitToDataBounds": False, "backgroundColor": "#ffffff",
                         "disableInteractive": False, "disableTooltipControl": False,
                         "hideToolbarOverlay": False, "hideLayerControl": False,
                         "hideViewControl": False, "initialLocation": "LAST_SAVED_LOCATION",
                         "showScaleControl": True, "showSpatialFilters": True,
                         "spatialFiltersAlpa": 0.3}}),
        "uiStateJSON": json.dumps({"isLayerTOCOpen": False, "openTOCDetails": []}),
    },
    "references": [
        {"name": "layer_2_source_index_pattern", "type": "index-pattern", "id": DV_TILES},
        {"name": "layer_3_source_index_pattern", "type": "index-pattern", "id": DV_ADMIN},
    ],
})
print(f"Carte definie — centre ({center_lat:.2f}, {center_lon:.2f}), zoom {map_zoom}.")

In [ ]:
# -----------------------------------------------------------------------------
#  9.5  Assemblage du dashboard (grille de 48 colonnes) + plage temporelle
# -----------------------------------------------------------------------------
HEADER_MD = (
    f"## {COUNTRY_NAME} — connectivite Ookla x population\n"
    f"**{' → '.join(QUARTER_LABELS)}** · reseau *{PRIMARY_NET}* · "
    f"{len(tiles):,} tuiles de ~610 m indexees en `geo_shape`\n\n"
    f"Utilisez le selecteur de dates en haut a droite pour filtrer l'ensemble des panneaux. "
    f"Sources : Ookla Open Data · WorldPop {WORLDPOP_YEAR} (1 km UN-adjusted) · "
    f"geoBoundaries ADM1. L'indice de fracture numerique est un composite pedagogique."
).replace(",", " ")

LAYOUT = [
    (f"{P}-kpi-tests",      "Tests de debit",                     0,  6, 12,  8),
    (f"{P}-kpi-dl",         f"Debit median ({PRIMARY_NET})",     12,  6, 12,  8),
    (f"{P}-kpi-pop",        "Population mesuree",                24,  6, 12,  8),
    (f"{P}-kpi-lat",        "Latence mediane",                   36,  6, 12,  8),
    (f"{P}-line-evo",       "Evolution trimestrielle du debit",   0, 14, 48, 14),
    (f"{P}-map",            "Carte de connectivite",              0, 28, 28, 19),
    (f"{P}-pie-speed",      "Population par classe de debit",    28, 28, 20, 19),
    (f"{P}-bar-dl",         "Debit median par region",            0, 47, 24, 16),
    (f"{P}-bar-ifn",        "Indice de fracture numerique",      24, 47, 24, 16),
    (f"{P}-bar-settlement", "Debit par type de peuplement",       0, 63, 20, 15),
    (f"{P}-bar-lat",        "Latence mediane par region",        20, 63, 28, 15),
    (f"{P}-table",          "Synthese par region",                0, 78, 48, 18),
]

panels, refs = [], []
panels.append({                       # en-tete markdown, embarque "by value"
    "version": ES_VERSION, "type": "visualization", "panelIndex": "0",
    "gridData": {"x": 0, "y": 0, "w": 48, "h": 6, "i": "0"},
    "embeddableConfig": {"savedVis": {
        "id": "", "title": "", "description": "", "type": "markdown",
        "params": {"fontSize": 11, "openLinksInNewTab": True, "markdown": HEADER_MD},
        "uiState": {}, "data": {"aggs": [], "searchSource": {}}},
        "hidePanelTitles": True, "enhancements": {}},
})

known = {o["id"] for o in objects}
for i, (so_id, title, x, y, w, h) in enumerate(LAYOUT, start=1):
    if so_id not in known:
        continue
    so_type = "map" if so_id.endswith("-map") else "visualization"
    panels.append({"version": ES_VERSION, "type": so_type, "panelIndex": str(i),
                   "gridData": {"x": x, "y": y, "w": w, "h": h, "i": str(i)},
                   "embeddableConfig": {"enhancements": {}},
                   "panelRefName": f"panel_{i}", "title": title})
    refs.append({"name": f"panel_{i}", "type": so_type, "id": so_id})

# Plage temporelle memorisee : du debut du premier trimestre a aujourd'hui.
first_period = min(tiles["period"])
dashboard = {
    "id": f"{P}-dashboard", "type": "dashboard",
    "attributes": {
        "title": DASHBOARD_TITLE,
        "description": (f"Genere par notebook — {COUNTRY_NAME}, {', '.join(QUARTER_LABELS)}, "
                        f"WorldPop {WORLDPOP_YEAR}. Charte AfDB-inspired Institutional."),
        "panelsJSON": json.dumps(panels),
        "optionsJSON": json.dumps({"useMargins": True, "syncColors": False,
                                   "syncCursor": True, "syncTooltips": False,
                                   "hidePanelTitles": False}),
        "timeRestore": True,
        "timeFrom": f"{first_period}T00:00:00.000Z",
        "timeTo": "now",
        "refreshInterval": {"pause": True, "value": 0},
        "version": 1,
        "kibanaSavedObjectMeta": {"searchSourceJSON": json.dumps(
            {"query": {"language": "kuery", "query": ""}, "filter": []})},
    },
    "references": refs,
}
print(f"Dashboard assemble : {len(panels)} panneaux, plage {first_period} → now.")

In [ ]:
# -----------------------------------------------------------------------------
#  9.6  Import objet par objet (+ NDJSON reutilisable)
#
#  Chaque visualisation est importee individuellement, puis le dashboard n'est
#  assemble qu'avec les panneaux reellement acceptes : un objet rejete ne fait
#  plus tomber tout le tableau de bord.
# -----------------------------------------------------------------------------
def kb_import(objs):
    nd = "\n".join(json.dumps(o, ensure_ascii=False) for o in objs)
    r = requests.post(f"{KIBANA_URL}/api/saved_objects/_import?overwrite=true",
                      headers={"kbn-xsrf": "true"},
                      files={"file": ("o.ndjson", nd, "application/ndjson")}, timeout=180)
    try:
        return r.json()
    except Exception:
        return {"success": False, "errors": [{"http": r.status_code, "body": r.text[:400]}]}

NDJSON_PATH = OUTDIR / f"dashboard_ookla_{ISO.lower()}.ndjson"
NDJSON_PATH.write_text("\n".join(json.dumps(o, ensure_ascii=False) for o in objects + [dashboard]),
                       encoding="utf-8")

ok_ids, failures = [], []
for o in objects:
    res = kb_import([o])
    if res.get("successCount"):
        ok_ids.append(o["id"])
    else:
        failures.append((o["id"], res.get("errors")))

keep = {r["name"] for r in refs if r["id"] in ok_ids}
dash_final = json.loads(json.dumps(dashboard))
dash_final["attributes"]["panelsJSON"] = json.dumps(
    [p for p in panels if p.get("panelRefName") is None or p["panelRefName"] in keep])
dash_final["references"] = [r for r in refs if r["id"] in ok_ids]
res_dash = kb_import([dash_final])

kpi_row([
    (f"{len(ok_ids)}/{len(objects)}", "Visualisations importees", AFDB["green"]),
    ("OUI" if res_dash.get("successCount") else "NON", "Dashboard cree",
     AFDB["deep"] if res_dash.get("successCount") else AFDB["brick"]),
    (f"{len(failures)}", "Objets rejetes", AFDB["brick"] if failures else AFDB["slate"]),
])
for oid, err in failures:
    print(f"\nRejete : {oid}\n  " + json.dumps(err, ensure_ascii=False)[:420])
if res_dash.get("errors"):
    print("\nErreur dashboard :", json.dumps(res_dash["errors"], ensure_ascii=False)[:500])
print(f"\nNDJSON reutilisable : {NDJSON_PATH}")

<div style="border-left:5px solid #00A86A;background:#F4F7F5;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#00704A;text-transform:uppercase;">10 &middot; ACC&Egrave;S</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Ouvrir le tableau de bord</div>
</div>

In [ ]:
# -----------------------------------------------------------------------------
#  10.1  Lien direct vers le dashboard
# -----------------------------------------------------------------------------
base = kibana_public_url(open_window=False) or KIBANA_PUBLIC
url = (base.rstrip("/") + f"/app/dashboards#/view/{P}-dashboard") if base else None

display(HTML(f"""
<div style="background:linear-gradient(135deg,#00553A 0%,#00704A 45%,#00A86A 100%);
            border-radius:14px;padding:26px 30px;font-family:Calibri,'Segoe UI',sans-serif;color:#fff;">
  <div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#F5C242;text-transform:uppercase;">DASHBOARD PR&Ecirc;T</div>
  <div style="font-size:26px;font-weight:700;margin-top:8px;">{DASHBOARD_TITLE}</div>
  <div style="font-size:14px;color:#E6F6EE;margin-top:10px;font-style:italic;">
    {('<a href="' + url + '" target="_blank" style="color:#F5C242;font-weight:700;">Ouvrir dans Kibana &rarr;</a>') if url else 'Utilisez un tunnel, ou importez le NDJSON dans votre propre Kibana.'}
  </div>
  <div style="height:5px;width:110px;background:#F5C242;border-radius:3px;margin-top:22px;"></div>
</div>"""))
if url:
    print(url)
    if IN_COLAB:
        try:
            from google.colab.output import serve_kernel_port_as_window
            serve_kernel_port_as_window(KIBANA_PORT)
        except Exception:
            pass

# --- Verification : ce qui existe reellement cote Kibana ----------------------
try:
    found = requests.get(f"{KIBANA_URL}/api/saved_objects/_find",
                         params={"type": ["dashboard", "visualization", "map"],
                                 "fields": "title", "per_page": 100},
                         headers={"kbn-xsrf": "true"}, timeout=60).json()
    objs = found.get("saved_objects", [])
    print(f"\n{len(objs)} saved objects presents :")
    for o in sorted(objs, key=lambda x: (x["type"], x["id"])):
        print(f"  {o['type']:<14} {o['id']:<30} {o['attributes'].get('title','')}")
except Exception as exc:
    print("Verification impossible :", exc)

In [ ]:
# -----------------------------------------------------------------------------
#  10.2  Exports complementaires (partage hors Kibana)
# -----------------------------------------------------------------------------
tiles_out = OUTDIR / f"ookla_tiles_{ISO.lower()}.parquet"
admin_out = OUTDIR / f"ookla_admin_{ISO.lower()}.csv"
tiles.to_parquet(tiles_out, index=False)
admin_df.to_csv(admin_out, index=False)

files = [(NDJSON_PATH, "Saved objects Kibana (dashboard + visualisations + carte)"),
         (tiles_out,   "Tuiles enrichies, tous trimestres (Parquet)"),
         (admin_out,   "Synthese region x reseau x trimestre (CSV)")]
rows = "".join(f"<tr><td style='padding:6px 12px;font-family:monospace;font-size:12px;'>{p.name}</td>"
               f"<td style='padding:6px 12px;font-size:13px;color:{AFDB['slate']};'>{d}</td>"
               f"<td style='padding:6px 12px;font-size:12px;text-align:right;'>{p.stat().st_size/1e6:,.2f} Mo</td></tr>"
               for p, d in files)
display(HTML(f"<table style=\"border-collapse:collapse;border:1px solid {AFDB['sage']};"
             f"border-radius:10px;overflow:hidden;font-family:Calibri,sans-serif;\">"
             f"<thead style='background:{AFDB['mint']};'><tr>"
             f"<th style='padding:8px 12px;text-align:left;color:{AFDB['deep']};font-size:11px;letter-spacing:2px;'>FICHIER</th>"
             f"<th style='padding:8px 12px;text-align:left;color:{AFDB['deep']};font-size:11px;letter-spacing:2px;'>CONTENU</th>"
             f"<th style='padding:8px 12px;text-align:right;color:{AFDB['deep']};font-size:11px;letter-spacing:2px;'>TAILLE</th>"
             f"</tr></thead><tbody>{rows}</tbody></table>"))
print("Repertoire :", OUTDIR)

<div style="border-left:5px solid #F5C242;background:#FDF4E0;border-radius:0 10px 10px 0;padding:14px 18px;font-family:Calibri,'Segoe UI',sans-serif;">
<div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#D49A00;text-transform:uppercase;">POUR ALLER PLUS LOIN</div>
<div style="font-size:22px;font-weight:700;color:#231F20;margin-top:4px;">Pistes pour la partie 2</div>
</div>

* **Data streams et ILM** &mdash; remplacer l'index unique par une *data stream* avec politique de cycle de vie&nbsp;: rollover trimestriel, passage en *warm* au bout d'un an.
* **Runtime fields** &mdash; calculer l'indice de fracture &agrave; la vol&eacute;e c&ocirc;t&eacute; cluster (Painless) plut&ocirc;t que de le figer &agrave; l'indexation&nbsp;: les pond&eacute;rations deviennent modifiables sans r&eacute;indexer.
* **Transforms** &mdash; l'API `_transform` d'Elasticsearch produit l'index d'agr&eacute;gation r&eacute;gionale sans passer par Python, et le tient &agrave; jour en continu.
* **Alerting** &mdash; r&egrave;gle Kibana sur la d&eacute;gradation trimestrielle du d&eacute;bit m&eacute;dian d'une r&eacute;gion.
* **Jointures enrichies** &mdash; croiser avec &eacute;coles et &eacute;tablissements de sant&eacute; (OpenStreetMap, Healthsites.io) via `geo_distance` pour mesurer la connectivit&eacute; des infrastructures publiques.
* **Vector tiles** &mdash; l'endpoint `_mvt` sert directement des tuiles vectorielles &agrave; une application cartographique maison.

In [ ]:
# -----------------------------------------------------------------------------
#  Nettoyage (decommenter pour liberer la machine)
# -----------------------------------------------------------------------------
# es.indices.delete(index=INDEX_TILES, ignore_unavailable=True)
# es.indices.delete(index=INDEX_ADMIN, ignore_unavailable=True)
# sh("pkill -f elasticsearch; pkill -f kibana")
# shutil.rmtree(STACKDIR, ignore_errors=True)

card("Session terminee",
     f"Cluster <b>{info['cluster_name']}</b> · index <code>{INDEX_TILES}</code> et "
     f"<code>{INDEX_ADMIN}</code> · {len(QUARTER_LABELS)} trimestres · dashboard "
     f"<b>{DASHBOARD_TITLE}</b>.<br>"
     "Les cellules de nettoyage sont comment&eacute;es volontairement&nbsp;: explorez d'abord "
     "Kibana, puis lib&eacute;rez les ressources.", tone="ok")